## **Final Project Introduction**
----
**Project name** : MACDHistogram-BB-ATR EventBased Backtest.  
**Group** : Traders 
**Date** : 4/2026  
**Summary** :
- **What is it about?**  
  A stateful trading strategy combining **MACD histogram**, **Bollinger Bands**, and **ATR** across 4 entry “legs” (Long Momentum, Long Reversion, Short Reversion, Short Momentum). It uses **robust scaling (Median/MAD)** for MACD to reduce noise, **log-space BB** for scale stability, and **ATR-based** TP/SL/trailing with lifecycle guards (`time_stop`, `cooldown`, `rebounce_block`).
- **Our Goal:**  
  Match the app research flow: tune on the in-sample window, then evaluate a yearly re-optimized walk-forward portfolio out-of-sample against SPY.

### **Data Inventory**
----
**Assets**: Top 10 large-cap U.S. stocks  
**Frequency**: Daily  
**Window**: [2020-01-01 → today]  
**In-sample period**: [2020-01-01 → 2024-12-31]  
**Out-of-sample period**: [2025-01-01 → today]  
**Portfolio optimization cadence**: Re-run at the start of each trade year  
**Sources**: Yahoo Finance


#**1. Download and Import Library**


In [ ]:
# @title Install library
!pip install optuna
!pip install ta
!pip install gurobipy


In [ ]:
# @title Import library
# =========================
# Standard Library Imports
# =========================
import argparse                # CLI arguments (e.g., --symbol, --start, --end)
import datetime as dt          # Date/time handling for ranges, timestamps
from datetime import datetime, timedelta
import logging                 # Logging (info/debug/errors)
import math                    # Math utilities (ceil/floor, isnan, etc.)
import operator
import os
import io
import base64
from dataclasses import dataclass  # Lightweight containers for params/config
from typing import Dict, List, Optional
import warnings

# =========================
# Third-Party Libraries
# =========================
from IPython.display import display
import numpy as np             # Numerical arrays, vectorized ops
import pandas as pd            # Tabular data handling (DataFrame/Series)
import matplotlib.cm as cm
import matplotlib.dates as mdates  # Plotting (equity curves, diagnostics)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import FuncFormatter, AutoMinorLocator
from matplotlib.dates import AutoDateLocator, DateFormatter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as st
from scipy.stats import norm as scipy_norm
import pytz                    # Timezone handling
import yfinance as yf          # Market data downloader (Yahoo Finance)
import optuna                  # Hyperparameter optimization/tuning
import gurobipy as gp          # Solve for tangent portfolio for portfolio selection
from sklearn.covariance import LedoitWolf

# =========================
# Technical Analysis Indicators
# =========================
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands, AverageTrueRange

# =========================
# Global Config
# =========================
warnings.filterwarnings("ignore")


#**2. Portfolio Selection**

### Modern Portfolio Optimizer — Tangent (Max Sharpe)

This section defines `ModernPortfolioOptimizerPro`, a Modern Portfolio Theory (MPT)–based optimizer that takes in **backtested strategy equity histories (strategy apply to the train period)** and produces a **tangent (max Sharpe) portfolio** plus its efficient frontier.

**Key features:**
- Uses **strategy-level daily returns** as inputs (not raw prices).
- Cleans and merges all strategies into a single return matrix, dropping near-zero variance series.
- Computes **mean returns** with exponentially weighted averages (recent data gets more weight).
- Estimates a **Ledoit–Wolf shrinkage covariance matrix** for more robust risk estimates.
- Adds **ridge (L2) regularization** to stabilize the covariance and improve numerical conditioning.
- Supports **annualization** of returns and covariances (assumes 252 trading days).
- Builds the **tangent portfolio**:
  - Uses the analytical max-Sharpe solution with a given risk-free rate.
  - Optionally enforces **long-only** weights by clipping negatives to zero and re-normalizing.
- Computes an **analytical efficient frontier**:
  - Uses closed-form formulas for the minimum-variance / efficient frontier under mean–variance assumptions.
  - Returns a DataFrame of points with expected return, volatility, and Sharpe ratio.
- Performs **allocation post-processing**:
  - Sorts assets by weight and optionally keeps only the **top_k**.
  - Renormalizes and rounds weights (2 decimals and sum portofilo = 100%), returning a clean stock → weight allocation dict.
- Provides visualization utilities:
  - `plot_efficient_frontier(...)` plots the efficient frontier with color‐coded Sharpe ratios.
  - Highlights both the **max-Sharpe point on the frontier grid** and the **selected tangent portfolio** found from the optimizer.
- The `run(...)` method wires everything together:
  1. Prepares the input matrix from strategy histories.
  2. Computes statistics (mean returns & covariance).
  3. Solves for the **tangent (max Sharpe) portfolio**.
  4. Computes the **efficient frontier**.
  5. Extracts the **final stock selections and allocations**.
  6. Stores the resulting portfolio in `self.portfolio_df` and prints a summary when `verbose=True`.


In [ ]:
# @title
# ============================================================
# 🧠 Modern Portfolio Optimizer — Tangent
# ============================================================
try:
    import gurobipy as gp
    _HAS_GUROBI = True
except Exception:
    _HAS_GUROBI = False


class ModernPortfolioOptimizerPro:
    """
    Modern Portfolio Theory (MPT) optimizer with:
      - Ledoit–Wolf shrinkage covariance
      - Ridge regularization for stability
      - Tangent (Max Sharpe) portfolio
      - Analytical efficient frontier (fast, exact)
      - Risk-free rate support
      - Auto-normalization & rounding
      - Portfolio metrics summary
    """

    def __init__(self, stock_equity_history, top_k=None,
                 weight_threshold=0.05, verbose=True):
        self.verbose = verbose
        self._strategy_hist = stock_equity_history
        self.symbols = sorted(stock_equity_history.keys())
        self.top_k = top_k
        self.weight_threshold = weight_threshold

        # Results
        self.mean_returns = None
        self.cov_matrix = None
        self.weights = None
        self.portfolio_df = None
        self.risk_free_rate = 0.0

    # ============================================================
    # Data Preparation
    # ============================================================
    def _prepare_inputs(self):
        dfs = []
        for s, hist in (self._strategy_hist or {}).items():
            if not hist:
                continue
            df = pd.DataFrame(hist, columns=["Date", "Price", "Equity", "StrategyRet", "StockRet"])
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df = df.dropna(subset=["Date"]).sort_values("Date")
            df = df.groupby("Date", as_index=True).agg({"StrategyRet": "sum"})
            dfs.append(df.rename(columns={"StrategyRet": s}))

        if not dfs:
            raise ValueError("No valid strategy histories available.")

        merged = pd.concat(dfs, axis=1, join="outer").fillna(0.0).astype(float)

        # Drop near-zero variance columns
        var = merged.var(axis=0)
        keep = var[var > 1e-12].index
        merged = merged[keep]

        if merged.shape[1] == 0:
            raise ValueError("All strategy series have near-zero variance.")

        if self.verbose:
            print(f"📦 Strategy return matrix built: {merged.shape} (days × symbols)")
        return merged

    # ============================================================
    # Compute Mean & Covariance
    # ============================================================
    def compute_statistics(self, data, annualize=True, ridge_alpha=1e-5, winsorize=True):
        """Compute mean returns & Ledoit–Wolf covariance."""
        if winsorize:
            data = data.clip(lower=data.quantile(0.01),
                             upper=data.quantile(0.99),
                             axis=1)

        # Exponential weighting for smoother mean returns
        weights = np.exp(np.linspace(-1, 0, len(data)))
        weights /= weights.sum()
        self.mean_returns = (data * weights[:, None]).sum(axis=0).values

        # Ledoit–Wolf shrinkage covariance
        lw = LedoitWolf().fit(data.values)
        self.cov_matrix = lw.covariance_

        # Annualize
        if annualize:
            TRADING_DAYS = 252
            self.mean_returns *= TRADING_DAYS
            self.cov_matrix *= TRADING_DAYS

        # Ridge regularization (L2)
        ridge = ridge_alpha * np.trace(self.cov_matrix) / len(self.cov_matrix)
        self.cov_matrix += ridge * np.eye(len(self.cov_matrix))

        if self.verbose:
            print(f"📈 Mean annualized return: {self.mean_returns.mean():.6f}")
            print(f"💠 Covariance matrix (after shrinkage): {self.cov_matrix.shape}")

    # ============================================================
    # Tangent (Max Sharpe) Portfolio
    # ============================================================
    def tangent_portfolio(self, risk_free_rate=0.0, long_only=True, lambda_reg=1e-3):
        mu = self.mean_returns
        Sigma = self.cov_matrix + lambda_reg * np.eye(len(self.cov_matrix))
        mu_excess = mu - risk_free_rate

        invS = np.linalg.pinv(Sigma)
        w = invS @ mu_excess

        if long_only:
            w = np.maximum(w, 0)

        w /= w.sum()
        self.weights = w
        self.risk_free_rate = risk_free_rate

        ret, vol, sharpe = self.portfolio_metrics(w)
        if self.verbose:
            print(f"🚀 Tangent Portfolio — Sharpe: {sharpe:.3f}")
            print(f"   Return: {ret:.4f}, Volatility: {vol:.4f}")
        return w

    # ============================================================
    # Portfolio Metrics
    # ============================================================
    def portfolio_metrics(self, w):
        mu, Sigma, rf = self.mean_returns, self.cov_matrix, self.risk_free_rate
        ret = mu @ w
        vol = np.sqrt(w @ Sigma @ w)
        sharpe = (ret - rf) / vol if vol > 0 else 0.0
        return ret, vol, sharpe

    # ============================================================
    # Analytical Efficient Frontier
    # ============================================================
    def efficient_frontier(self, num_points=30):
        mu, Sigma = self.mean_returns, self.cov_matrix
        invS = np.linalg.pinv(Sigma)
        ones = np.ones(len(mu))

        A = ones @ invS @ ones
        B = ones @ invS @ mu
        C = mu @ invS @ mu
        D = A * C - B**2

        target_returns = np.linspace(mu.min(), mu.max(), num_points)
        frontier = []

        for r in target_returns:
            w = invS @ ((C - B * r) * ones + (A * r - B) * mu) / D
            w = np.maximum(w, 0)
            w /= w.sum()
            ret, vol, sharpe = self.portfolio_metrics(w)
            frontier.append({"Return": ret, "Volatility": vol, "Sharpe": sharpe})

        df = pd.DataFrame(frontier)
        if self.verbose:
            print(f"✅ Efficient frontier computed analytically ({len(df)} points).")
        return df

    # ============================================================
    # Allocation Normalization & Rounding
    # ============================================================
    def get_allocations(self):
        df = pd.DataFrame({"Stock": self.symbols, "Weight": self.weights})
        df = df.sort_values("Weight", ascending=False)

        if self.top_k and self.top_k > 0:
            df = df.iloc[:self.top_k]
        else:
            df = df[df["Weight"].abs() > self.weight_threshold]

        total = df["Weight"].sum()
        df["Weight"] = (df["Weight"] / total).round(4)
        allocations = dict(zip(df["Stock"], df["Weight"]))

        if self.verbose:
            print(f"\n🎯 Selected {len(df)} active stocks:")
            for s, w in allocations.items():
                print(f"   {s}: {w:.4f} ({w*100:.2f}%)")
            print(f"✅ Total = {sum(allocations.values()):.4f}")

        return df["Stock"].tolist(), allocations

    # ============================================================
    # Plot Frontier
    # ============================================================
    def plot_efficient_frontier(self, frontier_df, highlight_portfolio=True,
                                show_max_sharpe=True):
        """
        Plot efficient frontier with:
          - Color = Sharpe ratio
          - Marker for Max Sharpe (on frontier grid)
          - Optional highlight for current self.weights
        """
        fig, ax = plt.subplots(figsize=(10, 6))

        sc = ax.scatter(
            frontier_df["Volatility"],
            frontier_df["Return"],
            c=frontier_df["Sharpe"],
            s=45,
            alpha=0.85,
            edgecolor="black",
            linewidths=0.4,
            label="Efficient frontier",
        )

        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label("Sharpe Ratio", rotation=90)

        # Max Sharpe point on discretized frontier
        if show_max_sharpe:
            ms_idx = frontier_df["Sharpe"].idxmax()
            ax.scatter(
                frontier_df.loc[ms_idx, "Volatility"],
                frontier_df.loc[ms_idx, "Return"],
                marker="^",
                s=90,
                label="Max Sharpe on frontier",
                zorder=6,
            )

        # Highlight current portfolio (tangent solution)
        if highlight_portfolio and self.weights is not None:
            ret, vol, sharpe = self.portfolio_metrics(self.weights)
            ax.scatter(
                vol,
                ret,
                marker="*",
                s=140,
                label=f"Selected portfolio (Sharpe={sharpe:.2f})",
                zorder=10,
            )

        ax.set_xlabel("Volatility (Annualized)")
        ax.set_ylabel("Expected Return (Annualized)")
        ax.set_title("Efficient Frontier — Modern Portfolio Optimizer Pro (Tangent Only)")
        ax.grid(alpha=0.25)
        ax.legend(loc="best")
        plt.tight_layout()
        plt.show()

    # ============================================================
    # Full Pipeline
    # ============================================================
    def run(self, mode="tangent", long_only=True, risk_free_rate=0.0):
        """
        Run full MPT optimization pipeline.

        mode is kept for backward compatibility, but whatever you pass,
        it will always compute the Tangent (Max Sharpe) portfolio.
        """
        data = self._prepare_inputs()
        self.compute_statistics(data)
        self.risk_free_rate = risk_free_rate

        self.tangent_portfolio(risk_free_rate=risk_free_rate, long_only=long_only)

        frontier = self.efficient_frontier()
        selected_stocks, allocations = self.get_allocations()

        self.portfolio_df = pd.DataFrame({
            "Stock": self.symbols,
            "Weight": self.weights
        }).sort_values("Weight", ascending=False).round(4)

        if self.verbose:
            print("\n📋 Final Portfolio Weights:")
            print(self.portfolio_df.to_string(index=False))
            print("=" * 70)

        return frontier, selected_stocks, allocations


#**3. Event-Based Backtesting**

In [ ]:
# @title Common Class
class Common_Class:
    def __init__(
        self,
        symbols,
        start,
        end,
        interval,
        capital,
        transaction_cost,
        allocations=None,
        verbose=True,
        rebalance_freq=None,
        warmup_days=150,
        leverage: float = 0.0,
        params = None
    ):
        self.symbols = list(symbols)
        self.start = start
        self.end = end
        self.interval = interval
        self.initial_capital = float(capital)
        self.warmup_days = warmup_days
        self.rebalance_freq = rebalance_freq
        self.leverage = float(leverage)

        if isinstance(start, str):
            start_date = pd.to_datetime(start)
        else:
            start_date = start

        self.data_start = (start_date - timedelta(days=warmup_days)).strftime('%Y-%m-%d')
        self.trading_start = start

        if allocations is None:
            self.allocations = {s: 1.0 / len(self.symbols) for s in self.symbols}
        else:
            total = sum(allocations.values())
            self.allocations = {s: v / total for s, v in allocations.items()}

        self.capital = {s: capital * self.allocations[s] for s in self.symbols}
        self.transaction_cost = float(transaction_cost)
        self.verbose = verbose

        # Portfolio state
        self.all_data = {}
        self.position = {s: 0 for s in self.symbols}
        self.quantity = {s: 0 for s in self.symbols}
        self.trades = {s: 0 for s in self.symbols}

        self.stored_data = pd.DataFrame(columns=[
            'trade', 'date', 'position', 'price', 'symbol', 'quantity',
            'capital', 'portfolio_value'
        ])
        self.portfolio_history = []
        self.portfolio_df = None
        self.stock_equity_history = {s: [] for s in self.symbols}
        self.stock_equity_df = None

        self.symbol = self.symbols[0] if len(self.symbols) == 1 else None
        self.data = None

        self.prepare_data()

    # ------------------------------------------------------------------
    def prepare_data(self):
        """Download and prepare OHLCV data for all symbols with warm-up period."""

        for symbol in self.symbols:
            try:
                stock = yf.Ticker(symbol)
                hist = stock.history(start=self.data_start, end=self.end, interval=self.interval)

                if hist.empty:
                    if self.verbose:
                        print(f"⚠️  No data for {symbol}, skipping.")
                    self.all_data[symbol] = pd.DataFrame()
                    continue

                df = pd.DataFrame(index=hist.index)
                df["Open"] = hist["Open"]
                df["High"] = hist["High"]
                df["Low"] = hist["Low"]
                df["Close"] = hist["Close"]
                df["Close_Price"] = hist["Close"]
                df["Volume"] = hist["Volume"]
                df["Return"] = np.log(df["Close"] / df["Close"].shift(1))
                df = df.dropna()


                if df.index.tz is not None:
                    df.index = df.index.tz_localize(None)

                self.all_data[symbol] = df

                if self.verbose:
                    warmup_bars = len(df[df.index < pd.to_datetime(self.trading_start)])
                    trading_bars = len(df[df.index >= pd.to_datetime(self.trading_start)])
                    print(f"✓ Loaded {len(df)} bars for {symbol} (warmup: {warmup_bars}, trading: {trading_bars})")

            except Exception as e:
                if self.verbose:
                    print(f"❌ Error loading {symbol}: {e}")
                self.all_data[symbol] = pd.DataFrame()

        if len(self.symbols) == 1:
            self.symbol = self.symbols[0]
            self.data = self.all_data[self.symbol]

    # ------------------------------------------------------------------
    def _get_date_price(self, bar, type="Open", symbol=None):
        """Get date and price at bar for symbol."""

        df = self.all_data[symbol]
        if df.empty or bar >= len(df):
            raise IndexError(f"Bar {bar} out of range for {symbol} (len={len(df)})")
        date = df.index[bar]
        if type in df.columns and pd.notna(df[type].iloc[bar]):
            price = float(df[type].iloc[bar])
            return date, price

        price_priority = ["Close", "Open", "High", "Low"]  # Fallback 1
        if type in price_priority:
            price_priority.remove(type)
            price_priority.insert(0, type)

        for price_type in price_priority:
            if price_type in df.columns and pd.notna(df[price_type].iloc[bar]):
                if self.verbose:
                    print(f"⚠️  {type} not available for {symbol} at bar {bar}, using {price_type}")
                price = float(df[price_type].iloc[bar])
                return date, price

        # Fallback 2: look backwards
        lookback = bar - 1
        while lookback >= 0:
            if type in df.columns and pd.notna(df[type].iloc[lookback]):
                date_fallback = df.index[lookback]
                price = float(df[type].iloc[lookback])
                if self.verbose:
                    print(f"⚠️  No price at bar {bar} for {symbol}, using {type} from bar {lookback} ({date_fallback})")
                return date, price

            for price_type in price_priority:
                if price_type in df.columns and pd.notna(df[price_type].iloc[lookback]):
                    date_fallback = df.index[lookback]
                    price = float(df[price_type].iloc[lookback])
                    if self.verbose:
                        print(f"⚠️  No price at bar {bar} for {symbol}, using {price_type} from bar {lookback} ({date_fallback})")
                    return date, price

            lookback -= 1

        raise IndexError(f"No valid price data found for {symbol} at or before bar {bar}")

    # ------------------------------------------------------------------
    def _record_stock_equity(self, bar_dict):
        """Record daily equity for each stock."""
        for s in self.symbols:
            df = self.all_data[s]
            bar = bar_dict.get(s, 0)

            if df.empty or bar >= len(df):
                continue

            date = df.index[bar]
            price = float(df["Close"].iloc[bar])

            cash = float(self.capital[s])
            position_value = float(self.quantity[s] * price)
            equity = cash + position_value

            history = self.stock_equity_history[s]

            if len(history) == 0:
                strategy_logreturn = 0.0
                stock_logreturn = 0.0
            else:
                _, prev_price, prev_equity, _, _ = history[-1]

                if prev_equity > 0 and equity > 0:
                    strategy_logreturn = np.log(equity / prev_equity)
                else:
                    strategy_logreturn = 0.0

                if prev_price > 0 and price > 0:
                    stock_logreturn = np.log(price / prev_price)
                else:
                    stock_logreturn = 0.0

            self.stock_equity_history[s].append((date, price, equity, strategy_logreturn, stock_logreturn))

    # ------------------------------------------------------------------
    def get_portfolio_value(self, bar):
        """
        Calculate total portfolio value.
        bar : dict or int
            If dict: {symbol: bar_index} for each symbol
            If int: uses same bar index for all symbols
        """
        total_value = 0.0

        if isinstance(bar, dict):
            bar_dict = bar
        else:
            bar_dict = {s: bar for s in self.symbols}

        for symbol in self.symbols:
            total_value += float(self.capital[symbol])
            df = self.all_data[symbol]

            if df.empty:
                continue

            b = bar_dict.get(symbol, 0)
            if b < 0 or b >= len(df):
                continue

            price = float(df["Close_Price"].iloc[b])
            qty = self.quantity[symbol]
            total_value += qty * price

        return float(total_value)

    # ------------------------------------------------------------------
    def buy_order(self, bar, symbol, quantity=None, dollar=None):
        date, price = self._get_date_price(bar + 1, "Open", symbol)

        # 1 Base quantity (no leverage)
        if quantity is None:
            if dollar is None:
                dollar = self.capital[symbol]

            cost_per_share = price * (1.0 + self.transaction_cost)
            base_qty = int(dollar / cost_per_share) if cost_per_share > 0 else 0
        else:
            base_qty = int(quantity)

        # 2 Apply leverage factor
        factor = 1.0 + max(self.leverage, 0.0)
        qty = int(base_qty * factor)

        cost = qty * price * (1.0 + self.transaction_cost)

        # 3 Validity checks
        if qty <= 0:
            if self.verbose:
                print(f"⚠️  BUY {symbol} skipped (qty={qty})")
            return

        # If no leverage, enforce no borrowing
        if self.leverage <= 0.0 and cost > self.capital[symbol]:
            if self.verbose:
                print(
                    f"⚠️  BUY {symbol} skipped (cost={cost:.2f} > cash={self.capital[symbol]:.2f})"
                )
            return

        # 4 Execute trade (capital may go negative with leverage > 0)
        self.capital[symbol] -= cost
        self.quantity[symbol] += qty
        self.trades[symbol] += 1

        # Update position state
        if self.quantity[symbol] > 0:
            self.position[symbol] = 1
        elif self.quantity[symbol] < 0:
            self.position[symbol] = -1
        else:
            self.position[symbol] = 0

        portfolio_value = self.get_portfolio_value(bar + 1)
        self._store_trade(
            trade=sum(self.trades.values()),
            date=date,
            position=1,
            price=price,
            symbol=symbol,
            quantity=qty,
            capital=self.capital[symbol],
            portfolio_value=portfolio_value,
        )

        if self.verbose:
            print(
                f"📈 BUY {qty} {symbol} @ ${price:.2f} | Cost: ${cost:.2f} | "
                f"Cash: ${self.capital[symbol]:.2f} | Portfolio: ${portfolio_value:.2f}"
            )

    # ------------------------------------------------------------------
    def sell_order(self, bar, symbol, last=False, quantity=None, dollar=None):
        if not last:
            date, price = self._get_date_price(bar + 1, "Open", symbol)
            held = self.quantity[symbol]
            if held <= 0:
                if self.verbose:
                    print(f"⚠️  SELL {symbol} skipped (no long position, qty={held})")
                return

            if quantity is None:
                if dollar is None:
                    qty = held
                else:
                    qty = int(dollar / price) if price > 0 else 0
                    qty = min(qty, held)
            else:
                qty = min(int(quantity), held)

            if qty <= 0:
                if self.verbose:
                    print(f"⚠️  SELL {symbol} skipped (qty={qty})")
                return

            proceeds = qty * price * (1.0 - self.transaction_cost)
            self.capital[symbol] += proceeds
            self.quantity[symbol] -= qty
            self.trades[symbol] += 1

            if self.quantity[symbol] > 0:
                self.position[symbol] = 1
            elif self.quantity[symbol] < 0:
                self.position[symbol] = -1
            else:
                self.position[symbol] = 0

            portfolio_value = self.get_portfolio_value(bar + 1)
        else:
            date, price = self._get_date_price(bar, "Close", symbol)
            held = self.quantity[symbol]
            if held <= 0:
                if self.verbose:
                    print(f"⚠️  SELL {symbol} skipped (no long position, qty={held})")
                return

            qty = held
            proceeds = qty * price * (1.0 - self.transaction_cost)
            self.capital[symbol] += proceeds
            self.quantity[symbol] -= qty
            self.trades[symbol] += 1
            self.position[symbol] = 0
            portfolio_value = self.get_portfolio_value(bar)

        self._store_trade(
            trade=sum(self.trades.values()),
            date=date,
            position=-1,
            price=price,
            symbol=symbol,
            quantity=qty,
            capital=self.capital[symbol],
            portfolio_value=portfolio_value,
        )

        if self.verbose:
            print(
                f"📉 SELL {qty} {symbol} @ ${price:.2f} | Proceeds: ${proceeds:.2f} | "
                f"Cash: ${self.capital[symbol]:.2f} | Portfolio: ${portfolio_value:.2f}"
            )

    # ------------------------------------------------------------------
    def short_order(self, bar, symbol, quantity=None, dollar=None):
        date, price = self._get_date_price(bar + 1, "Open", symbol)

        # 1 Base quantity (no leverage)
        if quantity is None:
            if dollar is None:
                dollar = self.capital[symbol]

            proceeds_per_share = price * (1.0 - self.transaction_cost)
            base_qty = int(dollar / proceeds_per_share) if proceeds_per_share > 0 else 0
        else:
            base_qty = int(quantity)

        # 2 Apply leverage
        factor = 1.0 + max(self.leverage, 0.0)
        qty = int(base_qty * factor)

        if qty <= 0:
            if self.verbose:
                print(f"⚠️  SHORT {symbol} skipped (qty={qty})")
            return

        proceeds = qty * price * (1.0 - self.transaction_cost)
        self.capital[symbol] += proceeds
        self.quantity[symbol] -= qty
        self.trades[symbol] += 1
        self.position[symbol] = -1

        portfolio_value = self.get_portfolio_value(bar + 1)
        self._store_trade(
            trade=sum(self.trades.values()),
            date=date,
            position=-1,
            price=price,
            symbol=symbol,
            quantity=-qty,
            capital=self.capital[symbol],
            portfolio_value=portfolio_value,
        )

        if self.verbose:
            print(
                f"🔻 SHORT {qty} {symbol} @ ${price:.2f} | "
                f"Proceeds: ${proceeds:.2f} | Cash: ${self.capital[symbol]:.2f} | "
                f"Portfolio: ${portfolio_value:.2f}"
            )

    # ------------------------------------------------------------------
    def cover_order(self, bar, symbol, last=False, quantity=None, dollar=None):
        if not last:
            date, price = self._get_date_price(bar + 1, "Open", symbol)

            held = self.quantity[symbol]
            if held >= 0:
                if self.verbose:
                    print(f"⚠️  COVER {symbol} skipped (no short position, qty={held})")
                return
            shorted_qty = abs(held)

            if quantity is None:
                if dollar is None:
                    qty = shorted_qty
                else:
                    qty = int(dollar / (price * (1.0 + self.transaction_cost))) if price > 0 else 0
                    qty = min(qty, shorted_qty)
            else:
                qty = min(int(quantity), shorted_qty)

            if qty <= 0:
                if self.verbose:
                    print(f"⚠️  COVER {symbol} skipped (qty={qty})")
                return

            cost = qty * price * (1.0 + self.transaction_cost)

            # Only block if NO leverage
            if self.leverage <= 0.0 and cost > self.capital[symbol]:
                if self.verbose:
                    print(
                        f"⚠️  COVER {symbol} skipped (insufficient capital: "
                        f"need ${cost:.2f}, have ${self.capital[symbol]:.2f})"
                    )
                return

            self.capital[symbol] -= cost
            self.quantity[symbol] += qty
            self.trades[symbol] += 1

            if self.quantity[symbol] > 0:
                self.position[symbol] = 1
            elif self.quantity[symbol] < 0:
                self.position[symbol] = -1
            else:
                self.position[symbol] = 0

            portfolio_value = self.get_portfolio_value(bar + 1)

        else:
            date, price = self._get_date_price(bar, "Close", symbol)
            held = self.quantity[symbol]
            if held >= 0:
                if self.verbose:
                    print(f"⚠️  COVER {symbol} skipped (no short position, qty={held})")
                return

            shorted_qty = abs(held)
            qty = shorted_qty
            cost = qty * price * (1.0 + self.transaction_cost)

            if self.leverage <= 0.0 and cost > self.capital[symbol]:
                if self.verbose:
                    print(
                        f"⚠️  COVER {symbol} skipped (insufficient capital: "
                        f"need ${cost:.2f}, have ${self.capital[symbol]:.2f})"
                    )
                return

            self.capital[symbol] -= cost
            self.quantity[symbol] += qty
            self.trades[symbol] += 1
            self.position[symbol] = 0

            portfolio_value = self.get_portfolio_value(bar)

        self._store_trade(
            trade=sum(self.trades.values()),
            date=date,
            position=1,
            price=price,
            symbol=symbol,
            quantity=qty,
            capital=self.capital[symbol],
            portfolio_value=portfolio_value,
        )

        if self.verbose:
            print(
                f"🔺 COVER {qty} {symbol} @ ${price:.2f} | Cost: ${cost:.2f} | "
                f"Cash: ${self.capital[symbol]:.2f} | Portfolio: ${portfolio_value:.2f}"
            )

    # ------------------------------------------------------------------
    def last_trade(self, bar_dict):
        for s in self.symbols:
            df = self.all_data[s]
            bar = bar_dict[s]
            if df.empty or bar >= len(df):
                continue

            qty = self.quantity[s]
            if qty == 0:
                continue

            if qty > 0:
                self.sell_order(bar, s, last=True, quantity=qty)
            else:
                self.cover_order(bar, s, last=True, quantity=abs(qty))

        if self.verbose:
            final_value = self.get_portfolio_value(bar_dict)
            roi = ((final_value - self.initial_capital) / self.initial_capital) * 100
            print("=" * 70)
            print("FINAL RESULTS")
            print(f"Initial Capital: ${self.initial_capital:,.2f}")
            print(f"Final Portfolio Value: ${final_value:,.2f}")
            print(f"Total Return: {roi:.2f}%")
            print(f"Total Trades: {sum(self.trades.values())}")
            print("=" * 70)

    # ------------------------------------------------------------------
    def close_all_positions(self, bar_dict):
        stock_values = {}
        for s in self.symbols:
            df = self.all_data[s]
            bar = bar_dict[s]
            if df.empty or bar >= len(df):
                continue

            qty = self.quantity[s]
            if qty == 0:
                continue

            if qty > 0:
                self.sell_order(bar - 1, s, quantity=qty)
            else:
                self.cover_order(bar - 1, s, quantity=abs(qty))

            alloc_weight = float(self.allocations.get(s, 1.0 / len(self.symbols)))
            init_cap = self.initial_capital * alloc_weight
            symbol_equity = self.capital[s]
            stock_return = ((symbol_equity - init_cap) / init_cap) * 100 if init_cap > 0 else 0.0

            stock_values[s] = {
                'initial_capital': init_cap,
                'final_value': symbol_equity,
                'return_pct': stock_return,
            }
        return stock_values

    # ------------------------------------------------------------------
    def rebalance(self, bar_dict, new_allocations=None):
        """Rebalance portfolio based on target allocations."""
        if self.verbose:
            print("\n" + "=" * 70)
            print("REBALANCING PORTFOLIO")
            print("=" * 70)

        total_cash = sum(self.capital.values())
        if self.verbose:
            print(f"\n💰 Total Cash After Closing: ${total_cash:,.2f}")

        if new_allocations is not None:
            total = sum(new_allocations.values())
            self.allocations = {s: v / total for s, v in new_allocations.items()}
            if self.verbose:
                print("\n New Allocations:")
                for s, w in self.allocations.items():
                    print(f"  {s}: {w:.2%}")

        for s in self.symbols:
            self.capital[s] = total_cash * self.allocations[s]
            self.quantity[s] = 0
            self.position[s] = 0

            if self.verbose:
                print(f"  {s}: ${self.capital[s]:,.2f} ({self.allocations[s]:.2%})")

        if self.verbose:
            portfolio_value = self.get_portfolio_value(bar_dict)
            print(f"\n Portfolio Value After Rebalance: ${portfolio_value:,.2f}")
            print("=" * 70 + "\n")

    # ------------------------------------------------------------------
    def _store_trade(self, trade, date, position, price, symbol, quantity, capital, portfolio_value):
        """Store trade details."""
        row = pd.DataFrame({
            'trade': [trade],
            'date': [pd.to_datetime(date)],
            'position': [position],
            'price': [float(price)],
            'symbol': [symbol],
            'quantity': [int(quantity)],
            'capital': [float(capital)],
            'portfolio_value': [float(portfolio_value)],
        })
        self.stored_data = pd.concat([self.stored_data, row], ignore_index=True)

    # ------------------------------------------------------------------
    def realised_balance(self, bar):
        """Print current cash balance."""
        date = str(self.all_data[self.symbols[0]].index[bar])[:10]
        cash = sum(self.capital[s] for s in self.symbols)
        print(f"💰 Date: {date} | Cash: ${cash:,.2f}")

    # ------------------------------------------------------------------
    def unrealised_balance(self, bar, symbol=None):
        """Print unrealized P&L (mark-to-market of positions)."""
        if symbol is None:
            cash = sum(self.capital.values())
            balance = 0.0
            for s in self.symbols:
                df = self.all_data[s]
                if df.empty or bar >= len(df):
                    continue
                price = float(df["Close_Price"].iloc[bar])
                balance += self.quantity[s] * price

            total_unrealised = (cash + balance) - self.initial_capital
            date = str(self.all_data[self.symbols[0]].index[bar])[:10]
            print(f" Date: {date} | Total Unrealized P&L: ${total_unrealised:,.2f}")
        else:
            df = self.all_data[symbol]
            if df.empty or bar >= len(df):
                print(f"⚠️  No data for {symbol}")
                return

            date, price = self._get_date_price(bar, "Close", symbol)
            date = str(date)[:10]
            position_value = self.quantity[symbol] * price
            symbol_initial = self.initial_capital * self.allocations[symbol]
            symbol_unrealised = (self.capital[symbol] + position_value) - symbol_initial

            print(f" Date: {date} | {symbol} Unrealized P&L: ${symbol_unrealised:,.2f}")

In [ ]:
# @title Buy and Hold Class


class BuyAndHold_Strategy(Common_Class):
    def run_strategy(self):
        """
          BuyAndHold_Strategy is a simple backtesting class that buys each symbol once
          on its first available trading day after trading_start, and then holds the
          position until the end of the backtest. After buying, it does not trade
          anymore; it only updates the portfolio value every day using the latest
          available prices. The result is a historical equity curve (portfolio_df)
          representing a pure buy-and-hold benchmark.
        """

        # Convert trading_start (string/date) into a Timestamp
        trading_start = pd.to_datetime(self.trading_start)

        # -------------------------------------------------
        # 1) ONE-TIME BUY — BUY EACH SYMBOL ONLY ONCE
        # -------------------------------------------------
        for s in self.symbols:
            df = self.all_data[s]
            if df.empty:
                continue  # no data → skip

            # Only use data after the actual trading_start (exclude warmup days)
            trade_df = df[df.index >= trading_start]
            if trade_df.empty:
                continue  # this symbol has no data after trading_start

            # First trading date of this symbol inside the trading window
            first_date = trade_df.index[0]

            # Index of that date in the original df
            first_bar = df.index.get_loc(first_date)

            # BUY ORDER LOGIC:
            # buy_order(bar) executes at bar+1 (open)
            # → we must send the buy signal at bar-1
            signal_bar = max(first_bar - 1, 0)
            self.buy_order(signal_bar, s)

        # -------------------------------------------------
        # 2) BUILD UNION OF ALL TRADING DATES
        #    (only dates >= trading_start)
        # -------------------------------------------------
        all_dates = sorted(
            set().union(*[
                df[df.index >= trading_start].index
                for df in self.all_data.values()
                if not df.empty
            ])
        )
        # all_dates = all days where at least one symbol has data

        # -------------------------------------------------
        # 3) COMPUTE PORTFOLIO VALUE OVER TIME
        #    Use the last-known bar for each symbol at each date
        # -------------------------------------------------
        for cur_date in all_dates:
            bar_dict = {}  # maps each symbol → bar index to use on this date

            for s, df in self.all_data.items():
                if df.empty:
                    continue

                # Find the last bar with date <= cur_date
                idx = df.index.searchsorted(cur_date, side="right") - 1

                if idx < 0:
                    continue  # this symbol has no price yet at this date

                bar_dict[s] = idx

            if not bar_dict:
                continue  # no prices available for this date → skip

            # Compute total portfolio value
            pv = self.get_portfolio_value(bar_dict)

            # Store date + portfolio value
            self.portfolio_history.append((cur_date, pv))

        # -------------------------------------------------
        # 4) BUILD THE FINAL PORTFOLIO DATAFRAME
        # -------------------------------------------------
        self.portfolio_df = (
            pd.DataFrame(self.portfolio_history, columns=["Date", "PortfolioValue"])
              .set_index("Date")
        )

        return self.portfolio_df

In [ ]:
# @title Backtest function
def run_multi_stock_backtest_unified(
    tickers, start, end, interval, initial_capital,
    strategy_class,
    tuning=False,
    show_each_stock=False,
    buyandhold = False,
    benchmark_ticker="SPY",
    benchmark_column="Adj Close",
    **kwargs
):
    """
    leverage (in kwargs) is interpreted as:
        per-trade size multiplier inside the strategy (via self.leverage).

    Examples:
        leverage = 0.0  -> normal position sizing (no leverage, default)
        leverage = 1.0  -> ~2x position size vs base sizing
        leverage = 2.0  -> ~3x position size, etc.
    """
    try:
        allocations = kwargs.get("allocations", None)

        # 1) Pull the params dict (MACD/BB/ATR..., etc.)
        params_dict = dict(kwargs.get("params", {}) or {})

        # 2) Extract rebalance_freq from params (and REMOVE it before passing on)
        rebalance_freq = params_dict.pop("rebalance_freq", 120)

        # 3) Per-trade leverage (NOT scaling initial_capital)
        leverage = kwargs.get("leverage", 0.0)   # default: no extra size

        # 4) Instantiate strategy with original initial_capital
        strategy = strategy_class(
            tickers,
            start,
            end,
            interval=interval,
            capital=initial_capital,                        # ⬅️ unchanged capital
            transaction_cost=kwargs.get("transaction_cost", 0.0),
            allocations=allocations,
            rebalance_freq=rebalance_freq,                 # from params_dict
            leverage=leverage,                             # ⬅️ pass to Common_Class / strategy
            verbose=kwargs.get("verbose", False),
            params=params_dict,                            # cleaned params

            # Pass through other kwargs that are not internal control flags
            **{
                k: v for k, v in kwargs.items()
                if k not in [
                    "transaction_cost",
                    "verbose",
                    "allocations",
                    "tuning",
                    "show_each_stock",
                    "rebalance_freq",
                    "params",
                    "leverage",           # don't forward leverage again
                    "benchmark_ticker",   # internal to this wrapper
                    "benchmark_column",
                    "risk_free_rate",     # for PerformanceMetrics, not strategy
                ]
            }
        )

        # 5) Run the strategy
        portfolio_df = strategy.run_strategy()

        if portfolio_df is None or portfolio_df.empty:
            print("⚠️  Warning: Empty portfolio data returned")
            return None, None, {}

        # 6) Per-stock performance summary
        allocs = strategy.allocations
        stock_performances = {}

        for s in strategy.symbols:
            df = strategy.all_data[s]
            if df.empty:
                stock_performances[s] = {
                    "initial_capital": 0.0,
                    "final_value": 0.0,
                    "return_pct": 0.0,
                    "trades": 0,
                }
                continue

            alloc_weight = float(allocs.get(s, 1.0 / len(strategy.symbols)))
            init_cap = strategy.initial_capital * alloc_weight  # true initial capital per stock
            symbol_equity = strategy.capital[s]  # run_strategy should have closed positions
            trades = int(strategy.trades.get(s, 0))
            stock_return = (
                ((symbol_equity - init_cap) / init_cap) * 100 if init_cap > 0 else 0.0
            )

            stock_performances[s] = {
                "initial_capital": init_cap,
                "final_value": symbol_equity,
                "return_pct": stock_return,
                "trades": trades,
            }

        # 7) Overall portfolio metrics & plotting (if not tuning)
        if not tuning:
            import yfinance as yf

            benchmark_df = yf.download(
                benchmark_ticker,
                start=start,
                end=end,
                interval=interval,
                progress=False,
            )

            if benchmark_df.empty:
                benchmark_series = None
            else:
                if benchmark_column in benchmark_df.columns:
                    benchmark_series = benchmark_df[benchmark_column]
                elif "Adj Close" in benchmark_df.columns:
                    benchmark_series = benchmark_df["Adj Close"]
                elif "Close" in benchmark_df.columns:
                    benchmark_series = benchmark_df["Close"]
                else:
                    benchmark_series = benchmark_df.iloc[:, 0]
            if buyandhold:
                metrics = PerformanceMetrics(
                    portfolio_df,
                    initial_capital=strategy.initial_capital)
            else:
                metrics = PerformanceMetrics(
                    portfolio_df,
                    initial_capital=strategy.initial_capital,  # = original initial_capital
                    benchmark_series=benchmark_series
                    # risk_free_rate=kwargs.get("risk_free_rate", 0.0)  # if you added this to PerformanceMetrics
                )
            metrics.summary(total_trades=sum(strategy.trades.values()))
            metrics.plot()

            if show_each_stock:
                print("\n" + "=" * 70)
                print("📊 PER-STOCK PERFORMANCE")
                print("=" * 70)
                for s, perf in stock_performances.items():
                    print(f"{s}:")
                    print(f"  Initial Capital: ${perf['initial_capital']:>12,.2f}")
                    print(f"  Final Value:     ${perf['final_value']:>12,.2f}")
                    print(f"  Return:          {perf['return_pct']:>12.2f}%")
                    print(f"  Total Trades:    {perf['trades']:>12}")
                    print("-" * 70)

        return portfolio_df, strategy, stock_performances

    except Exception as e:
        print(f"❌ Backtest failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None, {}

In [ ]:
# @title PerformanceMetrics
class PerformanceMetrics:
    def __init__(self, portfolio_df, initial_capital, benchmark_series=None, risk_free_rate=0.0):

        """
        portfolio_df: DataFrame with column 'PortfolioValue' and DateTimeIndex
        benchmark_series: optional Series of benchmark prices (e.g. SPY) aligned by date
        risk_free_rate: annual risk-free rate (e.g. 0.02 for 2%)
        """

        self.df = portfolio_df.copy()
        self.initial_capital = initial_capital
        self.risk_free_rate = risk_free_rate

        # Portfolio returns
        self.df["Return"] = self.df["PortfolioValue"].pct_change().fillna(0.0)

        # Optional benchmark
        self.benchmark = None
        if benchmark_series is not None:
            # Align on the same index, forward-fill missing if needed
            bench = benchmark_series.reindex(self.df.index).ffill()
            self.benchmark = bench
            self.df["BenchmarkReturn"] = bench.pct_change().fillna(0.0)
        else:
            self.df["BenchmarkReturn"] = np.nan

    # ========= ORIGINAL METRICS (unchanged behaviour) =========

    def sharpe(self, annualization_factor=252):
        std = self.df["Return"].std()
        if std <= 0:
            return np.nan
        # convert annual risk-free to per-period
        rf_daily = (1 + self.risk_free_rate) ** (1 / annualization_factor) - 1
        excess_ret = self.df["Return"] - rf_daily
        return np.sqrt(annualization_factor) * excess_ret.mean() / std

    def max_drawdown(self):
        roll_max = self.df["PortfolioValue"].cummax()
        dd = (self.df["PortfolioValue"] / roll_max) - 1.0
        return dd.min()

    def calmar(self):
        total_ret = self.df["PortfolioValue"].iloc[-1] / self.df["PortfolioValue"].iloc[0] - 1
        mdd = abs(self.max_drawdown())
        return total_ret / mdd if mdd != 0 else np.nan

    def total_return(self):
        return (self.df["PortfolioValue"].iloc[-1] / self.df["PortfolioValue"].iloc[0] - 1) * 100

    def cagr(self):
        """Calculate annualized return (CAGR)."""
        days = (self.df.index[-1] - self.df.index[0]).days
        years = days / 365.25
        if years <= 0:
            return 0.0
        return ((self.df["PortfolioValue"].iloc[-1] / self.initial_capital) ** (1 / years) - 1) * 100

    def volatility(self, annualization_factor=252):
        """Calculate annualized volatility."""
        return self.df["Return"].std() * np.sqrt(annualization_factor) * 100

    # ========= NEW METRICS =========

    def beta(self):
        """Portfolio beta vs benchmark."""
        if self.benchmark is None:
            return np.nan
        r_p = self.df["Return"]
        r_m = self.df["BenchmarkReturn"]
        cov = np.cov(r_p, r_m)[0, 1]
        var_m = np.var(r_m)
        return cov / var_m if var_m > 0 else np.nan

    def alpha(self, annualization_factor=252):
        """Jensen's alpha vs benchmark (annualized, in %)."""
        if self.benchmark is None:
            return np.nan
        beta = self.beta()
        if np.isnan(beta):
            return np.nan

        # average returns per period
        r_p = self.df["Return"].mean()
        r_m = self.df["BenchmarkReturn"].mean()

        rf_period = (1 + self.risk_free_rate) ** (1 / annualization_factor) - 1
        # expected portfolio return per period
        exp_rp = rf_period + beta * (r_m - rf_period)

        alpha_period = r_p - exp_rp
        alpha_annual = alpha_period * annualization_factor * 100
        return alpha_annual

    def sortino(self, annualization_factor=252):
        """Sortino ratio: uses downside deviation only."""
        rf_period = (1 + self.risk_free_rate) ** (1 / annualization_factor) - 1
        excess = self.df["Return"] - rf_period
        downside = excess[excess < 0]
        if len(downside) == 0:
            return np.nan
        downside_std = downside.std()
        if downside_std == 0:
            return np.nan
        return np.sqrt(annualization_factor) * excess.mean() / downside_std

    def omega_ratio(self, threshold=0.0):
        """
        Omega ratio at threshold (default 0 -> 0%).
        Ratio = sum of (returns - threshold)+ / sum of (threshold - returns)+
        """
        r = self.df["Return"]
        gains = np.clip(r - threshold, 0, None).sum()
        losses = np.clip(threshold - r, 0, None).sum()
        return gains / losses if losses > 0 else np.nan

    def ulcer_index(self):
        """
        Ulcer Index: sqrt( average( drawdown% ^ 2 ) ), drawdown% <= 0.
        """
        cummax = self.df["PortfolioValue"].cummax()
        drawdown_pct = (self.df["PortfolioValue"] / cummax - 1) * 100.0
        return np.sqrt(np.mean(drawdown_pct ** 2))

    def upside_potential_ratio(self, threshold=0.0):
        """
        Upside Potential Ratio (aka Ulcer Performance Index variant):
        mean(max(R - threshold, 0)) / sqrt(mean(min(R - threshold, 0)^2))
        """
        r = self.df["Return"]
        excess = r - threshold
        upside = np.clip(excess, 0, None)
        downside = np.clip(excess, None, 0)

        if (downside ** 2).mean() == 0:
            return np.nan
        return upside.mean() / np.sqrt((downside ** 2).mean())

    def skewness(self):
        """Skewness of daily returns."""
        return self.df["Return"].skew()

    def kurtosis(self):
        """Excess kurtosis of daily returns."""
        return self.df["Return"].kurtosis()

    def var(self, level=0.95):
        """
        Parametric (historical quantile) VaR at given confidence level.
        Returns % loss (positive number).
        """
        r = self.df["Return"].dropna()
        if len(r) == 0:
            return np.nan
        # loss is negative return; VaR is quantile of loss distribution
        var_ret = np.quantile(r, 1 - level)
        return -var_ret * 100  # in %

    def cvar(self, level=0.95):
        """
        Conditional VaR (Expected Shortfall) at given confidence level.
        Returns % loss (positive number).
        """
        r = self.df["Return"].dropna()
        if len(r) == 0:
            return np.nan
        var_ret = np.quantile(r, 1 - level)
        tail_losses = r[r <= var_ret]
        if len(tail_losses) == 0:
            return np.nan
        return -tail_losses.mean() * 100  # in %

    # ========= PLOT (unchanged except uses self.df["Return"]) =========

    def plot(self):
        """Plot equity curve and drawdown."""
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

        # Equity curve
        ax1.plot(self.df.index, self.df['PortfolioValue'], linewidth=2)
        ax1.set_title('Portfolio Equity Curve', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Portfolio Value ($)')
        ax1.grid(True, alpha=0.3)
        ax1.ticklabel_format(style='plain', axis='y')

        # Drawdown
        cummax = self.df['PortfolioValue'].cummax()
        drawdown = (self.df['PortfolioValue'] - cummax) / cummax * 100
        ax2.fill_between(self.df.index, drawdown, 0, alpha=0.3)
        ax2.plot(self.df.index, drawdown, linewidth=1)
        ax2.set_title('Drawdown (%)', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Drawdown (%)')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()


    def summary(self, total_trades=None):
        """Print formatted performance summary."""
        print("\n" + "=" * 70)
        print("📈 PERFORMANCE METRICS")
        print("=" * 70)
        print(f"Initial Capital:        ${self.initial_capital:>15,.2f}")
        print(f"Final Portfolio Value:  ${self.df['PortfolioValue'].iloc[-1]:>15,.2f}")
        print(f"Total Return:           {self.total_return():>15.2f}%")
        print(f"CAGR:                   {self.cagr():>15.2f}%")
        print(f"Annualized Volatility:  {self.volatility():>15.2f}%")
        print(f"Sharpe Ratio:           {self.sharpe():>15.2f}")
        print(f"Max Drawdown:           {self.max_drawdown():>15.2%}")
        print(f"Calmar Ratio:           {self.calmar():>15.2f}")

        # New: benchmark-relative metrics
        if self.benchmark is not None:
            print(f"Beta vs Benchmark:      {self.beta():>15.4f}")
            print(f"Alpha (annual, %):      {self.alpha():>15.2f}")

        # New: distribution / risk metrics
        print(f"Sortino Ratio:          {self.sortino():>15.2f}")
        print(f"Omega Ratio:            {self.omega_ratio():>15.2f}")
        print(f"Ulcer Index:            {self.ulcer_index():>15.2f}")
        print(f"UPI (UP Ratio):         {self.upside_potential_ratio():>15.2f}")
        print(f"Skewness:               {self.skewness():>15.4f}")
        print(f"Kurtosis:               {self.kurtosis():>15.4f}")
        print(f"VaR 95% (% loss):       {self.var():>15.2f}")
        print(f"CVaR 95% (% loss):      {self.cvar():>15.2f}")

        if total_trades is not None:
            print(f"Total Trades:           {total_trades:>15}")
        print("=" * 70 + "\n")

#**4. Trading Strategy Class**

## Trading Strategy Brief — MACD Histogram Fade + Bollinger Bands + ATR

### 1) Objective & Style
Bar-close, **stateful** system combining **momentum** and **mean-reversion** logic through four trading legs:
- **LM** (Long Momentum): follow upward trend continuation.
- **SM** (Short Momentum): follow downward trend continuation.
- **LR** (Long Reversion): buy oversold conditions near the lower Bollinger Band.
- **SR** (Short Reversion): sell overbought conditions near the upper Bollinger Band.

All exits are governed by **ATR-based take profit (TP)**, **stop loss (SL)**, and optional **trailing stops**. There is rebalancing every {rebalance_freq} trading days, it close all positions and redistribute the capital among the stocks.

---

### 2) Indicators & Notation — Plain-English Definitions

| Symbol | Full Name | What it Represents | Used For |
|---|---|---|---|
| **BB** | Bollinger Bands | Computed on **log-price** and then mapped back to price scale | Identify **overbought/oversold** conditions |
| **ub / mb / lb** | Upper / Middle / Lower Band | Upper band, middle mean band, lower band | Compare with Close for entry triggers |
| **h** | MACD Histogram | `MACD line − Signal line` | Captures short-term **momentum impulses** |
| **MACD_Z_Pos / Mid / Neg** | Robust MACD thresholds | Dynamic ± boundaries scaled by local histogram variability | Define “extreme” vs “near-zero” momentum regions |
| **MAD** | Median Absolute Deviation | Median of `|h − median(h)|` within rolling window | Robust dispersion measure resistant to outliers |
| **RS** | Robust Scale | `RS = 1.4826 × MAD(h)` | Robust estimate of histogram volatility |
| **ATR** | Average True Range | Rolling mean of True Range | Determines **TP/SL** and trailing distances |
| **pos** | Position | Strategy state (`−1` short / `0` flat / `+1` long) | Final position output |
| **TP / SL** | Take Profit / Stop Loss | Price levels at `ATR × multiplier` away from entry | Risk and exit control |
| **px** | Entry price | Close price at the moment of entry | Reference level for exits |
| **trail_mult** | Trailing multiplier | ATR multiplier for trailing stop/TP | Profit protection |
| **run_max / run_min** | Running extreme prices | Highest or lowest close since entry | Update trailing SL/TP levels |
| **cooldown** | Cooldown bars | Bars to wait after an exit | Avoid overtrading |
| **rebounce_block (rb)** | Flip guard | Bars blocking immediate opposite entry | Prevent whipsaws |
| **bars_in_trade** | Bars since entry | Lifespan of active position | Used for time stop logic |
| **bars_since_close / long / short** | Bars since last close or side entry | Entry gating and timing control |
| **buy_trend_counter / short_trend_counter** |Trend counter |Counters controlling how many momentum trades can occur consecutively| Limits consecutive LM/SM entries before reversion
| **rebalance_freq** | Rebalance period | Rebalance Every X trading days |
---

#### 2.1 Computation of **MACD_Z_Pos / MACD_Z_Mid / MACD_Z_Neg**

- **Input:** MACD Histogram `h(t)`
- **Rolling Window:** `W = macd_std_window`
- **Steps:**
  1. Compute rolling **MAD**: `MAD_h(t) = median{|h[i] − median(h)|}` over window `W`.
  2. Compute **Robust Scale**: `RS(t) = 1.4826 × MAD_h(t)`.
  3. Define adaptive thresholds:  
     - `MACD_Z_Pos = + macd_k × RS(t)`  
     - `MACD_Z_Mid =   macd_k_mid × RS(t)`  
     - `MACD_Z_Neg = − macd_k × RS(t)`

These thresholds expand and contract with histogram volatility, creating **context-aware** “momentum” and “extreme” zones for entries.

---

### 3) Entry Logic (evaluated at bar close)
- **SR**: SHORT if `h > MACD_Z_Pos` **and** `Close > ub`  
  (fading an overheated up move)
- **LR**: LONG if `h < MACD_Z_Neg` **and** `Close < lb`  
  (buying an oversold dip)
- **LM**: LONG if `0 ≤ h ≤ MACD_Z_Mid` **and** `Close > mb`  
  (following moderate positive momentum)
- **SM**: SHORT if `−MACD_Z_Mid ≤ h ≤ 0` **and** `Close < mb`  
  (following moderate negative momentum)

---

### 4) Gating & Risk Controls
- **Cooldown:** skip entry if `bars_since_close < cooldown`.
- **Flip Guard (`rebounce_block`):**  
  Block SHORT if `bars_since_long < rb`; block LONG if `bars_since_short < rb`.
- **Momentum Entry Cap:** LM/SM only allowed while trend counter `< 4` to avoid excessive re-entries.

---

### 5) On Entry
At entry price `px`:
- **LONG:**  
  `TP = px + ATR × tp_mult_*`,  
  `SL = px − ATR × sl_mult_*`
- **SHORT:**  
  `TP = px − ATR × tp_mult_*`,  
  `SL = px + ATR × sl_mult_*`
- Initialize `run_max = run_min = px`; reset counters.

---

### 6) Position Management & Exits
- **Hold logic:**  
  - LONG while `Close ≥ SL` and (TP inactive or `Close ≤ TP`)  
  - SHORT while `Close ≤ SL` and (TP inactive or `Close ≥ TP`)
- **Time stop:** close if `bars_in_trade ≥ time_stop`.
- **Breach:** on TP or SL violation, close and reset lifecycle.

---

### 7) Optional Trailing Stop
If `use_trailing`:
- **LONG:**  
  `run_max = max(run_max, Close)`  
  `SL = max(SL, run_max − ATR × trail_mult)`  
  if `trail_tp`: `TP = max(TP, run_max − ATR × trail_mult / 2)`
- **SHORT:**  
  `run_min = min(run_min, Close)`  
  `SL = min(SL, run_min + ATR × trail_mult)`  
  if `trail_tp`: `TP = min(TP, run_min + ATR × trail_mult / 2)`

---

### 8) Lifecycle Counters
Each bar updates:  
`bars_since_close`, `bars_since_long`, `bars_since_short`, and if in position → `bars_in_trade += 1`.  
Upon exit: reset `TP/SL/entry/run_max/run_min`, set `bars_since_close = 0`.

---

### 9) One-liner Summary
A bar-close trading system combining **MACD Histogram variability (via MAD × 1.4826 scaling)** with **Bollinger context** and **ATR-based exits**, balancing momentum following and mean-reversion while employing cooldowns and flip-guards to reduce noise.


In [ ]:
# @title MACD BB ATR Strategy Class
class MACD_BB_ATR_Strategy(Common_Class):
    def __init__(self, symbols, start, end, params=None, rebalance_freq=90, warmup_days=150, **kwargs):
        super().__init__(symbols, start, end, warmup_days=warmup_days, **kwargs)

        params = dict(params or {})
        if "rebalance_freq" in params:            # allow best_params to carry it
            rebalance_freq = params.pop("rebalance_freq")
        self.rebalance_freq = int(rebalance_freq)

        self.params = params if params else {
            "macd_slow": 26,
            "macd_fast": 12,
            "macd_signal": 9,
            "macd_std_window": 50,
            "macd_k": 2.0,
            "macd_k_mid": 0.5,
            "atr_window": 14,
            "bb_window": 20,
            "bb_std_dev": 2.0,
            "tp_mult_lm": 3.0,
            "sl_mult_lm": 1.5,
            "tp_mult_sm": 3.0,
            "sl_mult_sm": 1.5,
            "tp_mult_sr": 2.0,
            "sl_mult_sr": 1.0,
            "tp_mult_lr": 2.0,
            "sl_mult_lr": 1.0,
            "use_trailing": False,
            "trail_mult": 2.0,
            "trail_tp": False,
            "cooldown": 5,
            "rebounce_block": 10,
            "time_stop": 0
        }

        for s in self.symbols:
            df = self.all_data[s].copy()
            df = self._compute_indicators(df)
            self.all_data[s] = df

    def _compute_indicators(self, df):
        """Compute MACD, ATR, and Bollinger Bands indicators."""
        if df.empty:
            return df

        close = df["Close"]
        high = df["High"]
        low = df["Low"]

        macd = MACD(
            close=close,
            window_slow=int(self.params["macd_slow"]),
            window_fast=int(self.params["macd_fast"]),
            window_sign=int(self.params["macd_signal"]),
            fillna=False
        )
        df["MACD_Hist"] = macd.macd_diff()

        MAD_TO_STD = 1.4826  # Conversion factor from MAD to standard deviation

        def calc_mad(arr):
            if len(arr) == 0:
                return np.nan
            median_val = np.median(arr)
            mad = np.median(np.abs(arr - median_val))
            return mad

        rolling_mad = df["MACD_Hist"].rolling(
            int(self.params["macd_std_window"]),
            min_periods=int(self.params["macd_std_window"])
        ).apply(calc_mad, raw=False)

        robust_scale = MAD_TO_STD * rolling_mad
        df["MACD_Z_Pos"] = float(self.params["macd_k"]) * robust_scale
        df["MACD_Z_Mid"] = float(self.params["macd_k_mid"]) * robust_scale
        df["MACD_Z_Neg"] = -float(self.params["macd_k"]) * robust_scale

        atr = AverageTrueRange(
            high=high,
            low=low,
            close=close,
            window=int(self.params["atr_window"]),
            fillna=False
        )
        df["ATR"] = atr.average_true_range()

        log_price = np.log(df["Close"].where(df["Close"] > 0.0, np.nan)).astype(float)
        mean_log = log_price.rolling(
            int(self.params["bb_window"]),
            min_periods=int(self.params["bb_window"])
        ).mean()
        std_log = log_price.rolling(
            int(self.params["bb_window"]),
            min_periods=int(self.params["bb_window"])
        ).std(ddof=0)

        df["BB_Upper"] = np.exp(mean_log + float(self.params["bb_std_dev"]) * std_log)
        df["BB_Mid"] = np.exp(mean_log)
        df["BB_Lower"] = np.exp(mean_log - float(self.params["bb_std_dev"]) * std_log)

        return df

    def run_strategy(self):
        """Event-based backtest with warm-up period."""
        all_dates = sorted(set().union(*[df.index for df in self.all_data.values()]))

        # Split dates into warmup and trading periods
        trading_start_date = pd.to_datetime(self.trading_start)
        warmup_dates = [d for d in all_dates if d < trading_start_date]
        trading_dates = [d for d in all_dates if d >= trading_start_date]

        if self.verbose:
            print(f"\n📅 Trading Schedule:")
            print(f"   Data starts: {all_dates[0].strftime('%Y-%m-%d')}")
            print(f"   Warm-up period: {len(warmup_dates)} days")
            print(f"   Trading starts: {trading_start_date.strftime('%Y-%m-%d')}")
            print(f"   Trading days: {len(trading_dates)}")

        state = {}
        for s in self.symbols:
            state[s] = {
                'position': 0,
                'tp_price': np.nan,
                'sl_price': np.nan,
                'entry_price': np.nan,
                'run_max': np.nan,
                'run_min': np.nan,
                'buy_trend_counter': 0,
                'sell_trend_counter': 0,
                'bars_in_trade': 0,
                'bars_since_close': 10**9,
                'bars_since_long': 10**9,
                'bars_since_short': 10**9
            }

        days_since_rebalance = 0
        # Main event loop
        for cur_date in trading_dates:
            days_since_rebalance += 1
            if days_since_rebalance >= self.rebalance_freq:
                bar_dict = {s: df.index.get_loc(cur_date) if cur_date in df.index else 0
                           for s, df in self.all_data.items()}

                stock_values = self.close_all_positions(bar_dict)

                stock_strategy_daily_returns = {}
                for symbol, full_history in self.stock_equity_history.items():
                    # Get the last N entries where N = rebalance_freq
                    # full_history = [(date1, price1, equity1, strategy_logreturn1, stock_logreturn1), ...]

                    if len(full_history) <= self.rebalance_freq:
                        # If we have less data than rebalance_freq, use all of it
                        stock_strategy_daily_returns[symbol] = full_history
                    else:
                        stock_strategy_daily_returns[symbol] = full_history[-self.rebalance_freq:] # Slice to get only the last rebalance_freq entries
                # stock_strategy_daily_returns = {
                #     'AAPL': [(date1, price1, equity1, strategy_logreturn1, stock_logreturn1), ...],
                #     'MSFT': [ ... ],


                self.rebalance(bar_dict)
                # self.rebalance(bar_dict, new_allocations)

                for s in self.symbols:
                  state[s].update({
                      'position': 0,
                      'tp_price': np.nan,
                      'sl_price': np.nan,
                      'entry_price': np.nan,
                      'run_max': np.nan,
                      'run_min': np.nan,
                      'bars_in_trade': 0,
                      # Keep trend memory for continuity:
                      # 'buy_trend_counter': state[s]['buy_trend_counter'],
                      # 'sell_trend_counter': state[s]['sell_trend_counter'],
                      'bars_since_close': int(self.params["cooldown"]),
                      'bars_since_long': int(self.params["rebounce_block"]),
                      'bars_since_short': int(self.params["rebounce_block"])
                  })
                days_since_rebalance = 0

            for s in self.symbols:
                df = self.all_data[s]
                if cur_date not in df.index or df.empty:
                    continue

                bar = df.index.get_loc(cur_date)
                has_next = (bar + 1) < len(df)
                if bar < 1:  # Need previous bar
                    continue

                close_price = float(df["Close"].iloc[bar])
                upper_band = df["BB_Upper"].iloc[bar]
                mid_band = df["BB_Mid"].iloc[bar]
                lower_band = df["BB_Lower"].iloc[bar]
                macd_hist = df["MACD_Hist"].iloc[bar]
                macd_zpos = df["MACD_Z_Pos"].iloc[bar]
                macd_zmid = df["MACD_Z_Mid"].iloc[bar]
                macd_zneg = df["MACD_Z_Neg"].iloc[bar]
                atr_value = df["ATR"].iloc[bar]

                st = state[s]
                prev_pos = st['position']

                st['bars_since_close'] += 1
                st['bars_since_long'] += 1
                st['bars_since_short'] += 1
                st['bars_in_trade'] = (st['bars_in_trade'] + 1) if prev_pos != 0 else 0

                # Entry Conditions
                # SR: Short Reversion
                short_reversion = (np.isfinite([macd_hist, macd_zpos, upper_band]).all() and
                                macd_hist > macd_zpos and close_price > upper_band)

                # LR: Long Reversion
                long_reversion = (np.isfinite([macd_hist, macd_zneg, lower_band]).all() and
                                macd_hist < macd_zneg and close_price < lower_band)

                # LM: Long Momentum
                long_momentum = (np.isfinite([macd_hist, macd_zmid, mid_band]).all() and
                                0 <= macd_hist <= macd_zmid and close_price > mid_band)

                # SM: Short Momentum
                short_momentum = (np.isfinite([macd_hist, macd_zmid, mid_band]).all() and
                                -macd_zmid <= macd_hist <= 0 and close_price < mid_band)

                # Trailing Stop Updates
                if prev_pos != 0 and self.params["use_trailing"] and np.isfinite(atr_value):
                    if prev_pos == 1:  # Long
                        st['run_max'] = close_price if not np.isfinite(st['run_max']) else max(st['run_max'], close_price)
                        new_sl = st['run_max'] - float(self.params["trail_mult"]) * atr_value
                        st['sl_price'] = max(st['sl_price'], new_sl) if np.isfinite(st['sl_price']) else new_sl

                        if self.params["trail_tp"]:
                            new_tp = st['run_max'] - (float(self.params["trail_mult"]) / 2.0) * atr_value
                            st['tp_price'] = max(st['tp_price'], new_tp) if np.isfinite(st['tp_price']) else new_tp

                    else:  # Short
                        st['run_min'] = close_price if not np.isfinite(st['run_min']) else min(st['run_min'], close_price)
                        new_sl = st['run_min'] + float(self.params["trail_mult"]) * atr_value
                        st['sl_price'] = min(st['sl_price'], new_sl) if np.isfinite(st['sl_price']) else new_sl

                        if self.params["trail_tp"]:
                            new_tp = st['run_min'] + (float(self.params["trail_mult"]) / 2.0) * atr_value
                            st['tp_price'] = min(st['tp_price'], new_tp) if np.isfinite(st['tp_price']) else new_tp

                # Hold Existing Position
                if prev_pos == 1:  # Long
                    sl_ok = (not np.isfinite(st['sl_price'])) or (close_price >= st['sl_price'])
                    tp_ok = (not np.isfinite(st['tp_price'])) or (close_price <= st['tp_price'])

                    if sl_ok and tp_ok:
                        # Check time stop
                        if int(self.params["time_stop"]) > 0 and st['bars_in_trade'] >= int(self.params["time_stop"]):
                            if has_next: self.sell_order(bar, s)
                            st['position'] = 0
                            st['tp_price'] = st['sl_price'] = st['entry_price'] = np.nan
                            st['run_max'] = st['run_min'] = np.nan
                            st['bars_since_close'] = 0
                        continue
                    else:
                        # TP/SL breached - close position
                        if has_next: self.sell_order(bar, s)
                        st['position'] = 0
                        st['tp_price'] = st['sl_price'] = st['entry_price'] = np.nan
                        st['run_max'] = st['run_min'] = np.nan
                        st['bars_since_close'] = 0
                        continue

                elif prev_pos == -1:  # Short
                    sl_ok = (not np.isfinite(st['sl_price'])) or (close_price <= st['sl_price'])
                    tp_ok = (not np.isfinite(st['tp_price'])) or (close_price >= st['tp_price'])

                    if sl_ok and tp_ok:
                        # Check time stop
                        if int(self.params["time_stop"]) > 0 and st['bars_in_trade'] >= int(self.params["time_stop"]):
                            if has_next: self.cover_order(bar, s)
                            st['position'] = 0
                            st['tp_price'] = st['sl_price'] = st['entry_price'] = np.nan
                            st['run_max'] = st['run_min'] = np.nan
                            st['bars_since_close'] = 0
                        continue
                    else:
                        # TP/SL breached - close position
                        if has_next: self.cover_order(bar, s)
                        st['position'] = 0
                        st['tp_price'] = st['sl_price'] = st['entry_price'] = np.nan
                        st['run_max'] = st['run_min'] = np.nan
                        st['bars_since_close'] = 0
                        continue

                # Entry Logic (Flat Position)
                if prev_pos == 0:
                    # Cooldown check
                    if st['bars_since_close'] < int(self.params["cooldown"]):
                        continue

                    # Rebounce block
                    if int(self.params["rebounce_block"]) > 0:
                        if (short_reversion or short_momentum) and st['bars_since_long'] < int(self.params["rebounce_block"]):
                            continue
                        if (long_reversion or long_momentum) and st['bars_since_short'] < int(self.params["rebounce_block"]):
                            continue

                    # (1) Long Momentum Entry
                    if st['buy_trend_counter'] < 4 and long_momentum and np.isfinite(atr_value):
                        if has_next:
                            self.buy_order(bar, s)

                            if self.quantity[s] > 0:  # Entry successful
                                actual_entry = float(df["Open"].iloc[bar + 1])

                                st['position'] = 1
                                st['tp_price'] = actual_entry + atr_value * float(self.params["tp_mult_lm"])
                                st['sl_price'] = actual_entry - atr_value * float(self.params["sl_mult_lm"])
                                st['entry_price'] = actual_entry
                                st['run_max'] = st['run_min'] = actual_entry
                                st['buy_trend_counter'] += 1
                                st['bars_in_trade'] = 0
                                st['bars_since_long'] = 0
                        continue

                    # (2) Short Momentum Entry
                    if st['sell_trend_counter'] < 4 and short_momentum and np.isfinite(atr_value):
                        if has_next:
                            self.short_order(bar, s)

                            if self.quantity[s] < 0:  # Entry successful
                                actual_entry = float(df["Open"].iloc[bar + 1])

                                st['position'] = -1
                                st['tp_price'] = actual_entry - atr_value * float(self.params["tp_mult_sm"])
                                st['sl_price'] = actual_entry + atr_value * float(self.params["sl_mult_sm"])
                                st['entry_price'] = actual_entry
                                st['run_max'] = st['run_min'] = actual_entry
                                st['sell_trend_counter'] += 1
                                st['bars_in_trade'] = 0
                                st['bars_since_short'] = 0
                        continue

                    # (3) Short Reversion Entry
                    if st['buy_trend_counter'] > 1 and short_reversion and np.isfinite(atr_value):
                        if has_next:
                            self.short_order(bar, s)

                            if self.quantity[s] < 0:  # Entry successful
                                actual_entry = float(df["Open"].iloc[bar + 1])

                                st['position'] = -1
                                st['tp_price'] = actual_entry - atr_value * float(self.params["tp_mult_sr"])
                                st['sl_price'] = actual_entry + atr_value * float(self.params["sl_mult_sr"])
                                st['entry_price'] = actual_entry
                                st['run_max'] = st['run_min'] = actual_entry
                                st['bars_in_trade'] = 0
                                st['bars_since_short'] = 0
                        continue

                    # (4) Long Reversion Entry
                    if long_reversion and np.isfinite(atr_value):
                        if has_next:
                            self.buy_order(bar, s)

                            if self.quantity[s] > 0:  # Entry successful
                                actual_entry = float(df["Open"].iloc[bar + 1])

                                st['position'] = 1
                                st['tp_price'] = actual_entry + atr_value * float(self.params["tp_mult_lr"])
                                st['sl_price'] = actual_entry - atr_value * float(self.params["sl_mult_lr"])
                                st['entry_price'] = actual_entry
                                st['run_max'] = st['run_min'] = actual_entry
                                st['buy_trend_counter'] = 0
                                st['bars_in_trade'] = 0
                                st['bars_since_long'] = 0
                        continue

            # Record portfolio value at end of date
            bar_dict = {s: df.index.get_loc(cur_date) if cur_date in df.index else 0
                    for s, df in self.all_data.items()}
            self.portfolio_history.append((cur_date, self.get_portfolio_value(bar_dict)))
            self._record_stock_equity(bar_dict)

        # Close all positions at end
        final_bar_dict = {s: len(df) - 1 for s, df in self.all_data.items() if not df.empty}
        self.last_trade(final_bar_dict)

        # Create portfolio DataFrame
        self.portfolio_df = pd.DataFrame(
            self.portfolio_history,
            columns=["Date", "PortfolioValue"]
        ).set_index("Date")

        return self.portfolio_df


#**5. Tuning**

## Hyperparameter Tuning

**Goal.** Maximize **sharpe ratio** from backtests.

**Why Optuna?**
- **Efficient search:** smarter than grid/random via adaptive sampling.
- **Easy callbacks:** progress logging + early stopping.
- **Lightweight integration:** treat your backtest as a black-box objective.

**How it runs (simple).**
1. Sample a parameter set.
2. Run your backtest once.
3. Return **sharpe ratio**.
4. Repeat; keep the best.

**Objective.**
- `objective(params) → Sharp Ratio
- Study is set to **maximize** this value.

**Stopping.**
- Try up to 4000 trials, but **stop early** after 1000 trials with no improvement.

**What you get.**
- Best parameter set for **max Sharp Ratio**.
- A ranked history of trials for later analysis.


In [ ]:
# @title Tunning setup
class EarlyStoppingCallback:
    """Early stopping callback for Optuna single-objective studies."""

    def __init__(self, early_stopping_rounds: int, direction: str = "minimize") -> None:
        self.early_stopping_rounds = early_stopping_rounds
        self._iter = 0

        if direction == "minimize":
            self._operator = operator.lt
            self._score = np.inf
        elif direction == "maximize":
            self._operator = operator.gt
            self._score = -np.inf
        else:
            raise ValueError(f"Invalid direction: {direction}")

    def __call__(self, study: optuna.Study, trial: optuna.Trial) -> None:
        if self._operator(study.best_value, self._score):
            self._iter = 0
            self._score = study.best_value
        else:
            self._iter += 1

        if self._iter >= self.early_stopping_rounds:
            print(f"[EarlyStopping] No improvement for {self.early_stopping_rounds} trials. Stopping study.")
            study.stop()

def logging_callback(study: optuna.Study, frozen_trial: optuna.Trial) -> None:
    print(f"\rTrial now is: {frozen_trial.number}", end="", flush=True)
    if study.best_trial.number == frozen_trial.number:
        from datetime import datetime
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        v = frozen_trial.value
        print()
        print("=" * 60)
        print(f"[{ts}] 🎉 New Best Trial Found")
        print(f"Trial Number: {frozen_trial.number}")
        print(f"Value: {v:.6f}" if v is not None else "Value: None")
        for k, val in frozen_trial.params.items():
            print(f"  - {k}: {val}")
        print("=" * 60)


def objective(trial, tickers, start, end, initial_capital,
              allocations=None, leverage: float = 0.0):
    """Optuna objective: return Sharpe ratio."""

    params = {
        "macd_fast": trial.suggest_int("macd_fast", 10, 20),
        "macd_slow": trial.suggest_int("macd_slow", 35, 44),
        "macd_signal": trial.suggest_int("macd_signal", 5, 10),
        "macd_std_window": trial.suggest_int("macd_std_window", 70, 100),
        "macd_k": trial.suggest_float("macd_k", 1.0, 2.0, step=0.05),
        "macd_k_mid": trial.suggest_float("macd_k_mid", 0.5, 1.00, step=0.05),
        "bb_window": trial.suggest_int("bb_window", 14, 25),
        "bb_std_dev": trial.suggest_float("bb_std_dev", 1.4, 2.5, step=0.05),
        "atr_window": trial.suggest_int("atr_window", 10, 20),
        "tp_mult_lm": trial.suggest_float("tp_mult_lm", 3.5, 6.0, step=0.1),
        "sl_mult_lm": trial.suggest_float("sl_mult_lm", 1.0, 2.2, step=0.05),
        "tp_mult_sm": trial.suggest_float("tp_mult_sm", 3.0, 7.0, step=0.1),
        "sl_mult_sm": trial.suggest_float("sl_mult_sm", 1.0, 2.0, step=0.05),
        "tp_mult_lr": trial.suggest_float("tp_mult_lr", 8.0, 17.0, step=0.2),
        "sl_mult_lr": trial.suggest_float("sl_mult_lr", 2.0, 5.0, step=0.1),
        "tp_mult_sr": trial.suggest_float("tp_mult_sr", 3.0, 7.0, step=0.1),
        "sl_mult_sr": trial.suggest_float("sl_mult_sr", 1.5, 3.5, step=0.1),
        "use_trailing": trial.suggest_categorical("use_trailing", [True, False]),
        "trail_mult": trial.suggest_float("trail_mult", 1.5, 4.0, step=0.1),
        "trail_tp": trial.suggest_categorical("trail_tp", [True, False]),
        "time_stop": trial.suggest_int("time_stop", 0, 30),
        "cooldown": trial.suggest_int("cooldown", 0, 5),
        "rebounce_block": trial.suggest_int("rebounce_block", 0, 6),
        "rebalance_freq": trial.suggest_int("rebalance_freq", 30, 180, step=30),
    }

    # Validity guards
    if params["macd_signal"] > params["macd_fast"]:
        params["macd_signal"] = params["macd_fast"]

    if not (params["macd_fast"] < params["macd_slow"] and 8 <= params["macd_slow"] - params["macd_fast"] <= 22):
        return -1e9

    if params["macd_k_mid"] >= params["macd_k"]:
        return -1e9

    if params["tp_mult_lm"] < params["sl_mult_lm"]:
        return -1e9
    if params["tp_mult_sm"] < params["sl_mult_sm"]:
        return -1e9
    if params["tp_mult_lr"] <= params["sl_mult_lr"]:
        return -1e9
    if params["tp_mult_sr"] <= params["sl_mult_sr"]:
        return -1e9

    try:
        portfolio_df, strategy, stock_performances = run_multi_stock_backtest_unified(
            tickers, start, end,
            interval="1d",
            initial_capital=initial_capital,
            strategy_class=MACD_BB_ATR_Strategy,
            transaction_cost=0.0,
            params=params,
            rebalance_freq=params["rebalance_freq"],
            allocations=allocations,
            leverage=leverage,
            verbose=False,
            tuning=True
        )

        if portfolio_df is None or len(portfolio_df) < 60:
            return -1e9

        # Calculate Sharpe ratio (annualized)
        portfolio_df["Return"] = portfolio_df["PortfolioValue"].pct_change().fillna(0.0)

        mean_return = portfolio_df["Return"].mean()
        std_return = portfolio_df["Return"].std()

        if std_return <= 0 or not np.isfinite(std_return) or not np.isfinite(mean_return):
            return -1e9

        sharpe_ratio = np.sqrt(252) * mean_return / std_return

        if not np.isfinite(sharpe_ratio):
            return -1e9

        return float(sharpe_ratio)

    except Exception as e:
        print(f"Trial failed: {repr(e)}")
        return -1e9


def run_optimization(
    tickers,
    start,
    end,
    initial_capital,
    allocations=None,
    n_trials=4000,
    patience=1000,
    leverage: float = 0.0,   # 🔥 new argument
):
    """Run Optuna optimization for strategy parameters (at a given leverage)."""

    study = optuna.create_study(direction="maximize")

    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_rounds=patience,
        direction="maximize"
    )

    # Wrap objective to include allocations + leverage
    objective_with_params = lambda trial: objective(
        trial,
        tickers,
        start,
        end,
        initial_capital,
        allocations=allocations,
        leverage=leverage,
    )

    study.optimize(
        objective_with_params,
        n_trials=n_trials,
        callbacks=[early_stopping_callback, logging_callback]
    )

    print("\n" + "=" * 70)
    print("🎯 OPTIMIZATION RESULTS")
    print("=" * 70)
    print("Best parameters:", study.best_params)
    print(f"Best Sharpe ratio: {study.best_value:.4f}")
    print("=" * 70)

    return study.best_params, study.best_value

In [ ]:
# @title Tuning Run (In-Sample 2020-2024)
APP_DEFAULT_PARAMS = {
    "macd_fast": 20,
    "macd_slow": 35,
    "macd_signal": 6,
    "macd_std_window": 73,
    "macd_k": 2.0,
    "macd_k_mid": 0.8,
    "bb_window": 24,
    "bb_std_dev": 1.75,
    "atr_window": 19,
    "tp_mult_lm": 4.6,
    "sl_mult_lm": 1.7,
    "tp_mult_sm": 4.3,
    "sl_mult_sm": 1.95,
    "tp_mult_lr": 12.6,
    "sl_mult_lr": 2.2,
    "tp_mult_sr": 4.3,
    "sl_mult_sr": 3.1,
    "use_trailing": True,
    "trail_mult": 3.4,
    "trail_tp": True,
    "time_stop": 29,
    "cooldown": 1,
    "rebounce_block": 4,
    "rebalance_freq": 90,
}


print("OPTIMIZATION")
print("=" * 70)
print(f"Start time: {datetime.now()}")

tickers = ["AAPL", "AMZN", "META", "GOOG", "GOOGL", "NVDA", "MSFT", "AVGO", "TSLA", "BRK-B"]
start = "2020-01-01"
end = "2024-12-31"
initial_capital = 500000.0
transaction_cost = 0.00
leverage = 0.0

print("\n🔍 Starting parameter optimization on the in-sample window 2020-01-01 -> 2024-12-31...")
print(f"Fallback app-aligned params: {APP_DEFAULT_PARAMS}")

best_params, best_value = run_optimization(
    tickers,
    start,
    end,
    initial_capital,
    n_trials=4000,
    patience=1000,
    leverage=leverage,
)

print(f"\n✅ Best Sharpe in-sample: {best_value:.4f}")
print(f"⏰ End time: {datetime.now()}")


# **6. Walk-Forward Out-of-Sample Backtest (train: 2020-2024, test: 2025-today)**


In [ ]:
# @title In-Sample Sanity Check (2020-2024)
APP_DEFAULT_PARAMS = {
    "macd_fast": 20,
    "macd_slow": 35,
    "macd_signal": 6,
    "macd_std_window": 73,
    "macd_k": 2.0,
    "macd_k_mid": 0.8,
    "bb_window": 24,
    "bb_std_dev": 1.75,
    "atr_window": 19,
    "tp_mult_lm": 4.6,
    "sl_mult_lm": 1.7,
    "tp_mult_sm": 4.3,
    "sl_mult_sm": 1.95,
    "tp_mult_lr": 12.6,
    "sl_mult_lr": 2.2,
    "tp_mult_sr": 4.3,
    "sl_mult_sr": 3.1,
    "use_trailing": True,
    "trail_mult": 3.4,
    "trail_tp": True,
    "time_stop": 29,
    "cooldown": 1,
    "rebounce_block": 4,
    "rebalance_freq": 90,
}


if "best_params" not in globals():
    best_params = dict(APP_DEFAULT_PARAMS)
    print("Using APP_DEFAULT_PARAMS because the tuning cell was not executed.")

tickers = ["AAPL", "AMZN", "META", "GOOG", "GOOGL", "NVDA", "MSFT", "AVGO", "TSLA", "BRK-B"]
start = "2020-01-01"
end = "2024-12-31"
initial_capital = 500000.0
leverage = 0.0
transaction_cost = 0.00

print(f"🚀 BACKTESTING in-sample from {start} to {end}")

portfolio_insample, strat_insample, stock_performances_insample = run_multi_stock_backtest_unified(
    tickers,
    start,
    end,
    interval="1d",
    initial_capital=initial_capital,
    strategy_class=MACD_BB_ATR_Strategy,
    transaction_cost=transaction_cost,
    params=best_params,
    leverage=leverage,
    verbose=False,
    show_each_stock=False,
)

print("\n" + "=" * 70)
print(f"📊 Buy & Hold Strategy on SPY from {start} to {end}")
print("=" * 70)
portfolio_bh_insample, strat_bh_insample, stock_performances_bh_insample = run_multi_stock_backtest_unified(
    ["SPY"],
    start,
    end,
    interval="1d",
    initial_capital=initial_capital,
    strategy_class=BuyAndHold_Strategy,
    transaction_cost=transaction_cost,
    leverage=leverage,
    buyandhold=True,
    verbose=False,
)


In [ ]:
# @title Walk-Forward Helpers (2025 -> today, re-optimize every year)
APP_DEFAULT_PARAMS = {
    "macd_fast": 20,
    "macd_slow": 35,
    "macd_signal": 6,
    "macd_std_window": 73,
    "macd_k": 2.0,
    "macd_k_mid": 0.8,
    "bb_window": 24,
    "bb_std_dev": 1.75,
    "atr_window": 19,
    "tp_mult_lm": 4.6,
    "sl_mult_lm": 1.7,
    "tp_mult_sm": 4.3,
    "sl_mult_sm": 1.95,
    "tp_mult_lr": 12.6,
    "sl_mult_lr": 2.2,
    "tp_mult_sr": 4.3,
    "sl_mult_sr": 3.1,
    "use_trailing": True,
    "trail_mult": 3.4,
    "trail_tp": True,
    "time_stop": 29,
    "cooldown": 1,
    "rebounce_block": 4,
    "rebalance_freq": 90,
}



def build_walk_forward_windows(insample_start="2020-01-01", first_trade_year=2025, end_date=None):
    end_ts = pd.Timestamp(datetime.today().strftime("%Y-%m-%d") if end_date is None else end_date)
    windows = []

    for year in range(first_trade_year, end_ts.year + 1):
        train_start_ts = pd.Timestamp(insample_start)
        train_end_ts = pd.Timestamp(f"{year - 1}-12-31")
        trade_start_ts = pd.Timestamp(f"{year}-01-01")
        trade_end_ts = min(pd.Timestamp(f"{year}-12-31"), end_ts)

        if train_end_ts < train_start_ts or trade_start_ts > trade_end_ts:
            continue

        windows.append({
            "trade_year": year,
            "train_start": train_start_ts.strftime("%Y-%m-%d"),
            "train_end": train_end_ts.strftime("%Y-%m-%d"),
            "trade_start": trade_start_ts.strftime("%Y-%m-%d"),
            "trade_end": trade_end_ts.strftime("%Y-%m-%d"),
        })

    return windows


def run_yearly_walk_forward(
    tickers,
    params=None,
    insample_start="2020-01-01",
    first_trade_year=2025,
    end_date=None,
    initial_capital=500000.0,
    transaction_cost=0.0,
    leverage=0.0,
    top_k=6,
):
    params = dict(APP_DEFAULT_PARAMS if params is None else params)
    end_date = datetime.today().strftime("%Y-%m-%d") if end_date is None else end_date
    windows = build_walk_forward_windows(
        insample_start=insample_start,
        first_trade_year=first_trade_year,
        end_date=end_date,
    )

    if not windows:
        raise ValueError("No valid walk-forward windows were generated.")

    running_capital = float(initial_capital)
    combined_segments = []
    yearly_rows = []
    latest_optimizer = None
    latest_frontier = None
    latest_selected = None
    latest_allocations = None

    for window in windows:
        print("\n" + "=" * 70)
        print(
            f"📦 Trade year {window['trade_year']} | "
            f"train {window['train_start']} -> {window['train_end']} | "
            f"trade {window['trade_start']} -> {window['trade_end']}"
        )
        print("=" * 70)

        universe_df, strat_universe, _ = run_multi_stock_backtest_unified(
            tickers,
            window["train_start"],
            window["train_end"],
            interval="1d",
            initial_capital=running_capital,
            strategy_class=MACD_BB_ATR_Strategy,
            transaction_cost=transaction_cost,
            params=params,
            leverage=leverage,
            verbose=False,
            show_each_stock=False,
            tuning=True,
        )

        optimizer = ModernPortfolioOptimizerPro(
            stock_equity_history=strat_universe.stock_equity_history,
            top_k=top_k,
            verbose=True,
        )
        frontier, selected_tickers, allocations = optimizer.run(
            mode="tangent",
            risk_free_rate=0,
            long_only=True,
        )

        deployed_df, strat_deployed, _ = run_multi_stock_backtest_unified(
            selected_tickers,
            window["trade_start"],
            window["trade_end"],
            interval="1d",
            initial_capital=running_capital,
            strategy_class=MACD_BB_ATR_Strategy,
            transaction_cost=transaction_cost,
            params=params,
            leverage=leverage,
            allocations=allocations,
            verbose=False,
            show_each_stock=False,
            tuning=True,
        )

        if deployed_df is None or deployed_df.empty:
            raise ValueError(f"Empty deployed portfolio for trade year {window['trade_year']}")

        deployed_df = deployed_df.copy()
        deployed_df.index = pd.to_datetime(deployed_df.index)
        deployed_df = deployed_df[~deployed_df.index.duplicated(keep="last")].sort_index()

        start_capital_segment = running_capital
        end_capital_segment = float(deployed_df["PortfolioValue"].iloc[-1])
        running_capital = end_capital_segment
        combined_segments.append(deployed_df)

        yearly_rows.append({
            **window,
            "start_capital": round(start_capital_segment, 2),
            "end_capital": round(end_capital_segment, 2),
            "selected_count": len(selected_tickers),
            "selected_tickers": ", ".join(selected_tickers),
            "allocations": allocations,
        })

        latest_optimizer = optimizer
        latest_frontier = frontier
        latest_selected = selected_tickers
        latest_allocations = allocations

    portfolio_df = pd.concat(combined_segments)
    portfolio_df = portfolio_df[~portfolio_df.index.duplicated(keep="last")].sort_index()

    benchmark_df, benchmark_strat, _ = run_multi_stock_backtest_unified(
        ["SPY"],
        windows[0]["trade_start"],
        end_date,
        interval="1d",
        initial_capital=initial_capital,
        strategy_class=BuyAndHold_Strategy,
        transaction_cost=transaction_cost,
        buyandhold=True,
        verbose=False,
        tuning=True,
    )

    summary_df = pd.DataFrame(yearly_rows)
    latest_allocations_df = pd.DataFrame(
        [{"ticker": ticker, "weight": weight} for ticker, weight in (latest_allocations or {}).items()]
    ).sort_values("weight", ascending=False, ignore_index=True)

    return {
        "windows": windows,
        "portfolio_df": portfolio_df,
        "benchmark_df": benchmark_df,
        "benchmark_strat": benchmark_strat,
        "summary_df": summary_df,
        "latest_optimizer": latest_optimizer,
        "latest_frontier": latest_frontier,
        "latest_selected": latest_selected,
        "latest_allocations": latest_allocations,
        "latest_allocations_df": latest_allocations_df,
        "params": params,
    }


In [ ]:
# @title Run Walk-Forward Backtest (Out-of-Sample 2025 -> today)
APP_DEFAULT_PARAMS = {
    "macd_fast": 20,
    "macd_slow": 35,
    "macd_signal": 6,
    "macd_std_window": 73,
    "macd_k": 2.0,
    "macd_k_mid": 0.8,
    "bb_window": 24,
    "bb_std_dev": 1.75,
    "atr_window": 19,
    "tp_mult_lm": 4.6,
    "sl_mult_lm": 1.7,
    "tp_mult_sm": 4.3,
    "sl_mult_sm": 1.95,
    "tp_mult_lr": 12.6,
    "sl_mult_lr": 2.2,
    "tp_mult_sr": 4.3,
    "sl_mult_sr": 3.1,
    "use_trailing": True,
    "trail_mult": 3.4,
    "trail_tp": True,
    "time_stop": 29,
    "cooldown": 1,
    "rebounce_block": 4,
    "rebalance_freq": 90,
}


if "best_params" not in globals():
    best_params = dict(APP_DEFAULT_PARAMS)
    print("Using APP_DEFAULT_PARAMS because the tuning cell was not executed.")

tickers = ["AAPL", "AMZN", "META", "GOOG", "GOOGL", "NVDA", "MSFT", "AVGO", "TSLA", "BRK-B"]
initial_capital = 500000.0
transaction_cost = 0.00
leverage = 0.0
insample_start = "2020-01-01"
first_trade_year = 2025
walk_forward_end = datetime.today().strftime("%Y-%m-%d")

walk_forward_results = run_yearly_walk_forward(
    tickers=tickers,
    params=best_params,
    insample_start=insample_start,
    first_trade_year=first_trade_year,
    end_date=walk_forward_end,
    initial_capital=initial_capital,
    transaction_cost=transaction_cost,
    leverage=leverage,
    top_k=6,
)

walk_forward_portfolio_df = walk_forward_results["portfolio_df"]
walk_forward_benchmark_df = walk_forward_results["benchmark_df"]
walk_forward_summary = walk_forward_results["summary_df"]
walk_forward_latest_allocations_df = walk_forward_results["latest_allocations_df"]
walk_forward_latest_selected = walk_forward_results["latest_selected"]

print("\n✅ Walk-forward run completed.")
print(f"Out-of-sample window: 2025-01-01 -> {walk_forward_end}")
print(f"Latest selected tickers: {walk_forward_latest_selected}")
display(walk_forward_summary)
display(walk_forward_latest_allocations_df)


**Walk-forward with yearly portfolio selection and allocation**


In [ ]:
# @title Latest Annual Optimization Snapshot

summary_cols = [
    "trade_year",
    "train_start",
    "train_end",
    "trade_start",
    "trade_end",
    "start_capital",
    "end_capital",
    "selected_count",
    "selected_tickers",
]

display(walk_forward_summary[summary_cols])
print("\nLatest allocation used for the most recent trade year:")
display(walk_forward_latest_allocations_df)

latest_optimizer = walk_forward_results["latest_optimizer"]
latest_frontier = walk_forward_results["latest_frontier"]
if latest_optimizer is not None and latest_frontier is not None:
    latest_optimizer.plot_efficient_frontier(latest_frontier)


#**7. Measure Performance**

**Strategy Visualization Module**

**StrategyAdapter**  
- Extracts `PortfolioValue` from a completed strategy run.  
- Normalizes it to **growth of $1** and computes **daily returns**.  
- Provides `get_equity()` and `get_returns()` for plotting.

**PortfolioVisualizer**  
- Takes a strategy + benchmark, aligns dates, and normalizes both.  
- Computes:  
  - Daily returns & equity  
  - Drawdown  
  - Rolling 6-month return & volatility  
  - Monthly and yearly return stats

**plot_performance_dashboard()**  
Generates a 4-panel dashboard:  
1) Equity curve (strategy vs benchmark)  
2) Monthly returns (bar chart)  
3) Monthly heatmap  
4) Q–Q plot of monthly returns

**plot_risk_dashboard()**  
Generates a risk dashboard:  
1) Drawdown comparison  
2) Monthly return distribution  
3) Rolling return & vol (strategy)  
4) Rolling return & vol (benchmark)



In [ ]:
# @title Visualisation class
class StrategyAdapter:
    def __init__(self, strat, name: str | None = None, use_returns: bool = True):
        self.strat = strat
        self._name = name or strat.__class__.__name__
        self._use_returns = use_returns

        if not hasattr(strat, "portfolio_df"):
            raise ValueError("Run the strategy first; 'portfolio_df' not found.")

        eq = strat.portfolio_df["PortfolioValue"].copy().dropna()
        eq.index = pd.to_datetime(eq.index)
        eq = eq.sort_index()

        # Normalize to growth-of-$1 for plotting
        self._equity = eq / eq.iloc[0]
        self._returns = self._equity.pct_change().dropna()

    def get_name(self) -> str:
        return self._name

    def get_equity(self) -> pd.Series | None:
        return None if self._use_returns else self._equity

    def get_returns(self) -> pd.Series | None:
        return self._returns if self._use_returns else None

class PortfolioVisualizer:
    def __init__(
        self,
        strategy,
        benchmark_returns: pd.Series | None = None,
        benchmark_equity: pd.Series | None = None,
        benchmark_name: str = "Benchmark",
        title_prefix: str = "Strategy",
        trading_days: int = 252,
        *,
        theme: str = "plotly_white"
    ):
        self.title_prefix = title_prefix
        self.benchmark_name = benchmark_name
        self.TRADING_DAYS = trading_days
        self.theme = theme

        # ---- Strategy normalization ----
        s_ret = strategy.get_returns()
        s_eq  = strategy.get_equity()
        if s_eq is None and s_ret is None:
            raise ValueError("Strategy must provide returns or equity.")

        if s_eq is None:
            s_ret = pd.Series(s_ret).dropna()
            s_ret.index = pd.to_datetime(s_ret.index)
            s_eq = (1.0 + s_ret).cumprod()
        else:
            s_eq = pd.Series(s_eq).dropna()
            s_eq.index = pd.to_datetime(s_eq.index)
            s_ret = s_eq.pct_change().dropna()

        self.strategy_name = getattr(strategy, "get_name", lambda: "Strategy")()
        self.strat_ret = s_ret
        self.strat_eq  = s_eq

        # Benchmark normalization
        if benchmark_equity is None and benchmark_returns is None:
            raise ValueError("Provide benchmark_returns or benchmark_equity.")

        if benchmark_equity is None:
            b = pd.Series(benchmark_returns).dropna()
            b.index = pd.to_datetime(b.index)
            self.bm_ret = b
            self.bm_eq  = (1.0 + b).cumprod()
        else:
            e = pd.Series(benchmark_equity).dropna()
            e.index = pd.to_datetime(e.index)
            self.bm_eq = e
            self.bm_ret = e.pct_change().dropna()

        #align on common dates
        idx = self.strat_eq.index.intersection(self.bm_eq.index)
        self.strat_eq  = self.strat_eq.loc[idx]
        self.bm_eq     = self.bm_eq.loc[idx]
        self.strat_ret = self.strat_ret.loc[idx]
        self.bm_ret    = self.bm_ret.loc[idx]

        # risk & rolling
        self.strat_dd = self._drawdown_series(self.strat_eq)
        self.bm_dd    = self._drawdown_series(self.bm_eq)

        win = max(1, int(self.TRADING_DAYS * 0.5))  # ~6 months
        self.strat_roll_ret = (1 + self.strat_ret).rolling(win).apply(np.prod, raw=True) - 1.0
        self.strat_roll_vol = self.strat_ret.rolling(win).std(ddof=0) * np.sqrt(self.TRADING_DAYS)
        self.bm_roll_ret    = (1 + self.bm_ret).rolling(win).apply(np.prod, raw=True) - 1.0
        self.bm_roll_vol    = self.bm_ret.rolling(win).std(ddof=0) * np.sqrt(self.TRADING_DAYS)

        self.monthly_ret = self._monthly_returns(self.strat_ret)
        self.yearly_ret  = self._yearly_returns(self.strat_ret)

    @staticmethod
    def _monthly_returns(daily_ret: pd.Series) -> pd.Series:
        s = pd.Series(daily_ret).dropna()
        s.index = pd.to_datetime(s.index)
        return s.resample("ME").apply(lambda x: (1.0 + x).prod() - 1.0)

    @staticmethod
    def _yearly_returns(daily_ret: pd.Series) -> pd.Series:
        s = pd.Series(daily_ret).dropna()
        s.index = pd.to_datetime(s.index)
        y = s.resample("YE").apply(lambda x: (1.0 + x).prod() - 1.0)
        y.index = y.index.year
        return y

    @staticmethod
    def _drawdown_series(eq: pd.Series) -> pd.Series:
        roll_max = eq.cummax()
        return eq / roll_max - 1.0

    @staticmethod
    def _cagr(eq: pd.Series) -> float:
        days = (eq.index[-1] - eq.index[0]).days
        years = max(days, 1) / 365.25
        return float((eq.iloc[-1] / eq.iloc[0])**(1/years) - 1)

    @staticmethod
    def _maxdd(eq: pd.Series) -> float:
        dd = eq / eq.cummax() - 1.0
        return float(dd.min())

    def plot_performance_dashboard(self):
        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Equity Performance — Strategy vs Benchmark",
                f"Monthly Returns — {self.strategy_name}",
                "Monthly Returns Heatmap — Strategy",
                "Normal Q–Q — Strategy Monthly Returns",
            ),
            vertical_spacing=0.12,
            horizontal_spacing=0.08,
        )

        aligned = pd.concat(
            [
                self.strat_eq.rename(self.strategy_name),
                self.bm_eq.rename(self.benchmark_name),
            ],
            axis=1,
        ).dropna()
        aligned = aligned / aligned.iloc[0]  # growth of $1

        fig.add_trace(
            go.Scatter(
                x=aligned.index,
                y=aligned.iloc[:, 0],
                mode="lines",
                name=self.strategy_name,
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=aligned.index,
                y=aligned.iloc[:, 1],
                mode="lines",
                name=self.benchmark_name,
                line=dict(dash="dash"),
            ),
            row=1,
            col=1,
        )
        fig.update_yaxes(title_text="Growth of $1", row=1, col=1)

        mret = self.monthly_ret.dropna()
        fig.update_yaxes(title_text="Return", tickformat=".1%", row=1, col=2)

        if len(mret) > 0:
            colors = ["#2ca02c" if v > 0 else "#d62728" for v in mret.values]

            fig.add_trace(
                go.Bar(
                    x=mret.index,
                    y=mret.values,
                    marker_color=colors,
                    name="Monthly Return",
                ),
                row=1,
                col=2,
            )

            mean_val = float(mret.mean())
            fig.add_hline(
                y=mean_val,
                row=1,
                col=2,
                line_dash="dot",
                line_color="black",
                annotation_text=f"Mean: {mean_val*100:.2f}%",
                annotation_position="top left",
            )

        heat = self.monthly_ret.to_frame("ret").copy()

        if not heat.empty:
            heat["Year"] = heat.index.year
            heat["Month"] = heat.index.month

            pivot = heat.pivot(index="Year", columns="Month", values="ret").sort_index()
            pivot = pivot.reindex(columns=range(1, 13))  # months 1..12

            vals = (pivot.values * 100.0).astype(float)   # convert to %
            month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                            "Jul","Aug","Sep","Oct","Nov","Dec"]
            year_labels = [str(int(y)) for y in pivot.index.to_numpy()]

            text_vals = np.where(np.isnan(vals), "", np.round(vals, 2))

            fig.add_trace(
                go.Heatmap(
                    z=vals,
                    x=month_labels,
                    y=year_labels,
                    colorscale="RdYlGn",
                    zmid=0.0,
                    # data labels
                    text=text_vals,
                    texttemplate="%{text}%",
                    textfont=dict(size=9),
                    colorbar=dict(
                        title="Monthly Return (%)",
                        titleside="right",
                        x=-0.07,          # you can tweak this
                        xanchor="right",
                        y=0.25,
                        yanchor="middle",
                        len=0.60,
                        thickness=12,
                        tickfont=dict(size=10),
                        titlefont=dict(size=11),
                    ),
                    hovertemplate=(
                        "Year %{y}<br>"
                        "Month %{x}<br>"
                        "Return %{z:.2f}%<extra></extra>"
                    ),
                ),
                row=2,
                col=1,
            )

            fig.update_xaxes(title_text="Month", row=2, col=1)
            fig.update_yaxes(title_text="Year",  row=2, col=1)

        if len(mret) >= 2:
            # sort observed monthly returns
            y = np.sort(mret.values)

            # theoretical quantiles U ~ Uniform(0,1) then inverse normal CDF
            u = (np.arange(1, len(y) + 1) - 0.5) / len(y)

            # if you have SciPy, use scipy_norm.ppf(u)
            # otherwise assume you have _norm_ppf(u) defined somewhere
            try:
                x = self.scipy_norm.ppf(u)
            except AttributeError:
                x = _norm_ppf(u)  # your own approximation

            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=y,
                    mode="markers",
                    name="QQ Points",
                    marker=dict(size=6, color="#ff7f0e"),
                    hovertemplate=(
                        "Theoretical: %{x:.3f}<br>"
                        "Observed: %{y:.2%}<extra></extra>"
                    ),
                ),
                row=2,
                col=2,
            )

            lo = float(min(x.min(), y.min()))
            hi = float(max(x.max(), y.max()))

            fig.add_trace(
                go.Scatter(
                    x=[lo, hi],
                    y=[lo, hi],
                    mode="lines",
                    name="45°",
                    line=dict(dash="dash", width=1),
                    hoverinfo="skip",
                ),
                row=2,
                col=2,
            )

            fig.update_yaxes(
                tickformat=".0%",
                title_text="Observed Quantile",
                row=2,
                col=2,
            )
            fig.update_xaxes(
                title_text="Normal Quantile",
                row=2,
                col=2,
            )

        fig.update_layout(
            title=dict(
                text=f"{self.title_prefix} — Performance Dashboard",
                x=0.5,
                xanchor="center",
                y=0.98,
                yanchor="top",
            ),
            legend=dict(
                orientation="h",
                x=0.5,
                xanchor="center",
                y=1.06,          # > 1 => in the top margin
                yanchor="bottom",
            ),
            margin=dict(t=120),  # give space for title + legend
            hovermode="x unified",
            height=900,
        )


        fig.show()

    def plot_risk_dashboard(self):
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                "Drawdown — Strategy vs Benchmark",
                f"Distribution of Monthly Returns — {self.strategy_name}",
                f"Rolling 6-Month Return & Volatility — {self.strategy_name}",
                f"Rolling 6-Month Return & Volatility — {self.benchmark_name}",
            ),
            specs=[[{"type":"xy"}, {"type":"xy"}],
                   [{"type":"xy"}, {"type":"xy"}]],
            horizontal_spacing=0.09, vertical_spacing=0.12
        )

        # (1) Drawdown (area to zero)
        fig.add_trace(go.Scatter(
            x=self.strat_dd.index, y=self.strat_dd.values*100,
            name=f"{self.strategy_name} DD", mode="lines",
            fill="tozeroy", line=dict(width=1.8)
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=self.bm_dd.index, y=self.bm_dd.values*100,
            name=f"{self.benchmark_name} DD", mode="lines",
            fill="tozeroy", line=dict(width=1.3, dash="dot")
        ), row=1, col=1)
        fig.update_yaxes(title_text="Drawdown (%)", row=1, col=1)

        # (2) Monthly return distribution — FIXED (bars with mean line)
        mret = self.monthly_ret
        if len(mret):
            fig.add_trace(go.Histogram(
                x=mret.values * 100,
                nbinsx=15,
                name="Monthly Returns",
                marker=dict(color="rgba(50,150,255,0.6)",
                            line=dict(width=0.8, color="black")),
                hovertemplate="%{x:.2f}%<extra></extra>"
            ), row=1, col=2)

            mean_pct = float(mret.mean() * 100)
            fig.add_vline(
                x=mean_pct, row=1, col=2,
                line_dash="dash", line_width=1, line_color="black",
                annotation_text=f"Mean: {mean_pct:.2f}%",
                annotation_position="top right"
            )
            fig.update_xaxes(title_text="Monthly Return (%)", row=1, col=2)
            fig.update_yaxes(title_text="Frequency", row=1, col=2)

        # (3) Rolling — Strategy
        fig.add_trace(go.Scatter(
            x=self.strat_roll_ret.index, y=self.strat_roll_ret.values*100,
            name="Rolling Return (6m)", mode="lines", line=dict(width=1.8)
        ), row=2, col=1)
        fig.add_trace(go.Scatter(
            x=self.strat_roll_vol.index, y=self.strat_roll_vol.values*100,
            name="Rolling Vol (6m)", mode="lines", line=dict(width=1.4)
        ), row=2, col=1)
        fig.update_yaxes(title_text="Percent", row=2, col=1)

        # (4) Rolling — Benchmark
        fig.add_trace(go.Scatter(
            x=self.bm_roll_ret.index, y=self.bm_roll_ret.values*100,
            name="Rolling Return (6m)", mode="lines", line=dict(width=1.8)
        ), row=2, col=2)
        fig.add_trace(go.Scatter(
            x=self.bm_roll_vol.index, y=self.bm_roll_vol.values*100,
            name="Rolling Vol (6m)", mode="lines", line=dict(width=1.4)
        ), row=2, col=2)
        fig.update_yaxes(title_text="Percent", row=2, col=2)

        fig.update_layout(
            template=self.theme,
            title=dict(
                text=f"<b>{self.title_prefix} — Risk Dashboard</b>",
                x=0.5,
                xanchor="center",
                y=0.98,
                yanchor="top",
            ),
            legend=dict(
                orientation="h",
                x=0.5,
                xanchor="center",
                y=1.06,          # right below the header, in the margin
                yanchor="bottom",
            ),
            margin=dict(t=120),
            hovermode="x unified",
            height=800,
            font=dict(family="Helvetica Neue, Arial", size=12),
            title_font=dict(size=18),
            hoverlabel=dict(bgcolor="white", font_size=11),
        )


        fig.show()

def _norm_ppf(p):
    a1=-3.969683028665376e+01; a2= 2.209460984245205e+02
    a3=-2.759285104469687e+02; a4= 1.383577518672690e+02
    a5=-3.066479806614716e+01; a6= 2.506628277459239e+00
    b1=-5.447609879822406e+01; b2= 1.615858368580409e+02
    b3=-1.556989798598866e+02; b4= 6.680131188771972e+01
    b5=-1.328068155288572e+01
    c1=-7.784894002430293e-03; c2=-3.223964580411365e-01
    c3=-2.400758277161838e+00; c4=-2.549732539343734e+00
    c5= 4.374664141464968e+00; c6= 2.938163982698783e+00
    d1= 7.784695709041462e-03; d2= 3.224671290700398e-01
    d3= 2.445134137142996e+00; d4= 3.754408661907416e+00
    plow = 0.02425; phigh = 1 - plow
    p = np.asarray(p)
    q = np.zeros_like(p, dtype=float)

    mask = p < plow
    if np.any(mask):
        ql = np.sqrt(-2*np.log(p[mask]))
        q[mask] = (((((c1*ql + c2)*ql + c3)*ql + c4)*ql + c5)*ql + c6) / \
                  ((((d1*ql + d2)*ql + d3)*ql + d4)*ql + 1)

    mask = (p >= plow) & (p <= phigh)
    if np.any(mask):
        r = p[mask] - 0.5
        t = r*r
        q[mask] = (((((a1*t + a2)*t + a3)*t + a4)*t + a5)*t + a6)*r / \
                  (((((b1*t + b2)*t + b3)*t + b4)*t + b5)*t + 1)

    mask = p > phigh
    if np.any(mask):
        qu = np.sqrt(-2*np.log(1-p[mask]))
        q[mask] = -(((((c1*qu + c2)*qu + c3)*qu + c4)*qu + c5)*qu + c6) / \
                    ((((d1*qu + d2)*qu + d3)*qu + d4)*qu + 1)
    return q

In [ ]:
# @title Visualize Walk-Forward Strategy vs SPY Benchmark

class PortfolioFrameWrapper:
    def __init__(self, portfolio_df):
        self.portfolio_df = portfolio_df

walk_forward_adapter = StrategyAdapter(
    PortfolioFrameWrapper(walk_forward_portfolio_df),
    name="MACD+BB+ATR Walk-Forward",
    use_returns=True,
)

spy_equity = walk_forward_benchmark_df["PortfolioValue"].copy().dropna()
spy_equity.index = pd.to_datetime(spy_equity.index)
spy_ret = spy_equity.pct_change().dropna()

viz = PortfolioVisualizer(
    strategy=walk_forward_adapter,
    benchmark_returns=spy_ret,
    benchmark_name="SPY Buy & Hold",
    title_prefix="MACD+BB+ATR Walk-Forward vs SPY",
)

viz.plot_performance_dashboard()
viz.plot_risk_dashboard()


## Walk-Forward Notes

- The strategy parameters are fit on the in-sample window `2020-01-01 -> 2024-12-31`.
- The out-of-sample evaluation starts on `2025-01-01` and runs through the current day.
- Portfolio selection and allocation are re-optimized at the start of each trade year using all data from `2020-01-01` through the prior year-end.
- The combined walk-forward equity curve is compared against SPY buy-and-hold over the same out-of-sample dates.


# **8. Walk-Forward Review Area**
### Workflow now matches the app-style research flow

1. Run the tuning cell on the in-sample window `2020-01-01 -> 2024-12-31`.
2. Run the walk-forward cell to backtest `2025-01-01 -> today`.
3. The notebook will re-run portfolio optimization at the start of each trade year.
4. Use the summary and visualization cells below to inspect yearly allocations and the combined equity curve.


In [ ]:
# @title Walk-Forward Summary Table

review_cols = [
    "trade_year",
    "train_start",
    "train_end",
    "trade_start",
    "trade_end",
    "start_capital",
    "end_capital",
    "selected_tickers",
]

display(walk_forward_summary[review_cols])


In [ ]:
# @title Walk-Forward Headline Metrics

def compute_headline_metrics(portfolio_df):
    equity = portfolio_df["PortfolioValue"].copy().dropna()
    equity.index = pd.to_datetime(equity.index)
    total_return = equity.iloc[-1] / equity.iloc[0] - 1.0
    daily_ret = equity.pct_change().dropna()
    cagr = (equity.iloc[-1] / equity.iloc[0]) ** (365.25 / max((equity.index[-1] - equity.index[0]).days, 1)) - 1.0
    drawdown = equity / equity.cummax() - 1.0
    sharpe = np.sqrt(252) * daily_ret.mean() / daily_ret.std(ddof=0) if daily_ret.std(ddof=0) > 0 else np.nan
    return {
        "total_return": total_return,
        "cagr": cagr,
        "max_drawdown": drawdown.min(),
        "sharpe": sharpe,
    }

strategy_metrics = compute_headline_metrics(walk_forward_portfolio_df)
benchmark_metrics = compute_headline_metrics(walk_forward_benchmark_df)

headline_df = pd.DataFrame([
    {"series": "MACD+BB+ATR Walk-Forward", **strategy_metrics},
    {"series": "SPY Buy & Hold", **benchmark_metrics},
])

for col in ["total_return", "cagr", "max_drawdown", "sharpe"]:
    if col != "sharpe":
        headline_df[col] = headline_df[col].map(lambda x: f"{x:.2%}")
    else:
        headline_df[col] = headline_df[col].map(lambda x: f"{x:.2f}" if pd.notna(x) else "nan")

display(headline_df)


In [ ]:
# @title Latest Selected Tickers and Allocations

print("Latest selected tickers:")
print(walk_forward_results["latest_selected"])
print("\nLatest portfolio weights:")
display(walk_forward_latest_allocations_df)
